# Hindcast 0008-01 Tropospheric Wave / Ozone Tests

Purpose: reproduce the `Hindcast_vertical_analysis.ipynb` O3 RMSE vs winter-wave-activity figure with the cleaned Hindcast products, then test which EP-flux wave component and which tropospheric signals matter for `0008-01`.

This notebook writes only under:

```text
/home/weiji/restart_exam/code_cleaned/Hindcast_experiment/TEST_TROPOS/outputs/0008-01
```

Data are read from `/mnt/soclim0/public_data/weiji`. Run with the `jimnew` environment.

In [ ]:
from __future__ import annotations

import glob
import math
import os
import re
import warnings
from pathlib import Path
from typing import Dict, Iterable, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
from scipy.stats import linregress, pearsonr

plt.rcParams.update({
    "figure.facecolor": "white",
    "savefig.facecolor": "white",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.grid": True,
    "grid.linestyle": "--",
    "grid.alpha": 0.35,
})

CASE = "0008-01"
REF_YEAR = 8

REPO_ROOT = Path("/home/weiji/restart_exam/code_cleaned")
TEST_ROOT = REPO_ROOT / "Hindcast_experiment" / "TEST_TROPOS"
OUT_ROOT = TEST_ROOT / "outputs" / CASE
FIG_DIR = OUT_ROOT / "figures"
TABLE_DIR = OUT_ROOT / "tables"
CACHE_DIR = OUT_ROOT / "cache"
for d in [FIG_DIR, TABLE_DIR, CACHE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DATA_ROOT = Path("/mnt/soclim0/public_data/weiji")
HINDCAST_ROOT = DATA_ROOT / "Hindcast"
BWCN_ROOT = DATA_ROOT / "BWCN"
B2000_ROOT = DATA_ROOT / "B2000WCN001002_timefixed"

CASE_ROOT = HINDCAST_ROOT / CASE

# Main windows. The EP window matches the old Hindcast_vertical_analysis scatter.
O3_END = (5, 30)
EP_WINDOW = ((1, 20), (2, 10))
EP_WINDOW_LABEL = "Jan20-Feb10"
AO_BASE_WINDOW = EP_WINDOW
AO_BASE_WINDOW_LABEL = EP_WINDOW_LABEL
AO_TEST_WINDOWS = {
    "Jan20-Feb10": EP_WINDOW,
    "Jan": ((1, 1), (1, 31)),
    "Jan-Feb": ((1, 1), (2, 28)),
    "Jan-Mar": ((1, 1), (3, 31)),
}
Z300_WINDOW = EP_WINDOW
Z300_WINDOW_LABEL = EP_WINDOW_LABEL
MONTH_WINDOWS = {
    "Jan": ((1, 1), (1, 31)),
    "Feb": ((2, 1), (2, 28)),
    "Mar": ((3, 1), (3, 31)),
    "Apr": ((4, 1), (4, 30)),
    "May": ((5, 1), (5, 30)),
}
MONTH_ORDER = ["Jan", "Feb", "Mar", "Apr", "May"]

LAT_EP = (40.0, 80.0)
LAT_POLAR = (60.0, 90.0)
LAT_Z300 = (20.0, 90.0)
PLEV_EP_PA = 10000.0
PLEV_EP_HPA = PLEV_EP_PA / 100.0
PLEV_U_PA = 1000.0
PLEV_Z300_PA = 30000.0
PLEV_Z300_HPA = PLEV_Z300_PA / 100.0

EP_SCALAR_DESCRIPTION = (
    f"mean -ep2 vertical EP-flux component at {PLEV_EP_HPA:.0f} hPa, "
    f"cos-lat mean {LAT_EP[0]:.0f}-{LAT_EP[1]:.0f}N, {EP_WINDOW_LABEL}; not EP-flux divergence"
)
Z300_PATTERN_DESCRIPTION = (
    f"{PLEV_Z300_HPA:.0f} hPa height, {Z300_WINDOW_LABEL} mean, "
    f"{LAT_Z300[0]:.0f}-{LAT_Z300[1]:.0f}N, zonal mean removed before pattern metrics"
)

WAVES = ["all_waves", "wave1", "wave2", "wave_rest"]
WAVE_LABELS = {
    "all_waves": "All waves",
    "wave1": "Wave 1",
    "wave2": "Wave 2",
    "wave_rest": "Wave rest",
    "wave1_plus_wave2": "Wave 1 + Wave 2",
}

# Expensive sections cache their results. Keep True for the full requested test.
RUN_Z300_DIAGNOSTICS = True
BUILD_U60N10_IF_MISSING = True
CLIM_MAX_B2000_YEARS_FOR_Z300 = None  # None = all B2000WCN001002 Z3 years available.

print(f"Output root: {OUT_ROOT}")
print(f"Case root exists: {CASE_ROOT.exists()} -> {CASE_ROOT}")

In [ ]:
# -----------------------------
# Shared helpers
# -----------------------------

def member_short_id(member) -> str:
    text = str(member)
    m = re.search(r"\.(\d{3})\.cam\.h3", text)
    if m:
        return m.group(1)
    m = re.search(r"\.(\d{3})\.", text)
    return m.group(1) if m else text


def date_parts(date_values):
    arr = np.asarray(date_values, dtype=np.int64)
    year = arr // 10000
    mmdd = arr % 10000
    month = mmdd // 100
    day = mmdd % 100
    return year, month, day


def date_mask(date_values, start=(1, 1), end=(5, 30), year: Optional[int] = None):
    yy, mm, dd = date_parts(date_values)
    start_key = start[0] * 100 + start[1]
    end_key = end[0] * 100 + end[1]
    key = mm * 100 + dd
    mask = (key >= start_key) & (key <= end_key)
    if year is not None:
        mask = mask & (yy == int(year))
    return mask


def one_dim_date(ds_or_da) -> np.ndarray:
    date = ds_or_da["date"]
    if "member" in date.dims:
        date = date.isel(member=0)
    return np.asarray(date.values, dtype=np.int64)


def assign_member_short_coord(da: xr.DataArray) -> xr.DataArray:
    if "member" not in da.dims:
        return da
    mids = [member_short_id(v) for v in da["member"].values]
    return da.assign_coords(member_short=("member", mids))


def select_latband(da: xr.DataArray, lat_range: Tuple[float, float], lat_name="lat") -> xr.DataArray:
    lat = da[lat_name]
    descending = float(lat.values[0]) > float(lat.values[-1])
    lo, hi = lat_range
    return da.sel({lat_name: slice(hi, lo) if descending else slice(lo, hi)})


def coslat_mean(da: xr.DataArray, lat_range: Optional[Tuple[float, float]] = None, lat_name="lat") -> xr.DataArray:
    if lat_range is not None:
        da = select_latband(da, lat_range, lat_name=lat_name)
    weights = np.cos(np.deg2rad(da[lat_name])).clip(0, 1)
    return da.weighted(weights.fillna(0)).mean(lat_name, skipna=True)


def finite_corr(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 3:
        return np.nan, np.nan
    r, p = pearsonr(x[mask], y[mask])
    return float(r), float(p)


def scatter_fit(ax, df, xcol, ycol, title, xlabel=None, ylabel=None, color="tab:blue", annotate_members=True):
    sub = df[[xcol, ycol, "member"]].replace([np.inf, -np.inf], np.nan).dropna()
    ax.scatter(sub[xcol], sub[ycol], s=62, color=color, edgecolor="black", linewidth=0.5, alpha=0.88)
    if annotate_members:
        for _, row in sub.iterrows():
            ax.text(row[xcol], row[ycol], str(row["member"]), fontsize=7, alpha=0.65)
    if len(sub) >= 3:
        fit = linregress(sub[xcol].values, sub[ycol].values)
        xx = np.linspace(sub[xcol].min(), sub[xcol].max(), 100)
        ax.plot(xx, fit.slope * xx + fit.intercept, color="crimson", ls="--", lw=1.8)
        ax.text(
            0.03, 0.97,
            f"R = {fit.rvalue:.3f}\nP = {fit.pvalue:.2e}",
            transform=ax.transAxes,
            va="top",
            ha="left",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.84, edgecolor="0.7"),
        )
    ax.set_title(title)
    ax.set_xlabel(xlabel or xcol)
    ax.set_ylabel(ylabel or ycol)
    return ax


def savefig(fig, name):
    png = FIG_DIR / f"{name}.png"
    pdf = FIG_DIR / f"{name}.pdf"
    fig.savefig(png, dpi=300, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    print(f"Saved: {png}")
    return png, pdf


def mmdd_label(date_values):
    _, mm, dd = date_parts(date_values)
    return np.array([f"{int(m):02d}-{int(d):02d}" for m, d in zip(mm, dd)])


def month_ticks(date_values):
    _, mm, dd = date_parts(date_values)
    positions, labels = [], []
    names = {1:"Jan", 2:"Feb", 3:"Mar", 4:"Apr", 5:"May", 6:"Jun"}
    seen = set()
    for i, (m, d) in enumerate(zip(mm, dd)):
        if int(d) == 1 and int(m) not in seen:
            positions.append(i)
            labels.append(names.get(int(m), str(int(m))))
            seen.add(int(m))
    return positions, labels


def standardize_1d(y):
    y = np.asarray(y, dtype=float)
    return (y - np.nanmean(y)) / np.nanstd(y)

In [ ]:
# -----------------------------
# Data loaders for the cleaned Hindcast products
# -----------------------------

def load_hindcast_o3(case=CASE, pressure_tag="30_70hPa"):
    path = HINDCAST_ROOT / case / "partial_O3" / f"{case}_partial_O3_all_ranges_members.nc"
    if not path.exists():
        raise FileNotFoundError(f"Missing cleaned partial O3 product: {path}")
    with xr.open_dataset(path, decode_times=False) as ds:
        var = f"O3_partial_60_90N_{pressure_tag}"
        da = assign_member_short_coord(ds[var]).load()
        date = one_dim_date(ds)
    mask = date_mask(date, start=(1, 1), end=O3_END)
    da = da.isel(lead_time=mask)
    date = date[mask]
    da = da.assign_coords(date=("lead_time", date))
    return da, date


def load_bwcn_ref_o3(year=REF_YEAR, pressure_tag="30_70hPa"):
    path = BWCN_ROOT / "partial_O3" / "BWCN_partial_O3_all_ranges.nc"
    if not path.exists():
        raise FileNotFoundError(path)
    with xr.open_dataset(path, decode_times=False) as ds:
        var = f"O3_partial_60_90N_{pressure_tag}"
        date = np.asarray(ds["date"].values, dtype=np.int64)
        mask = date_mask(date, start=(1, 1), end=O3_END, year=year)
        da = ds[var].isel(time=mask).load()
        date = date[mask]
    da = da.rename({"time": "lead_time"}).assign_coords(lead_time=np.arange(len(date)), date=("lead_time", date))
    return da, date


def load_epflux_wave(case=CASE, wave="all_waves", plev_pa=PLEV_EP_PA, lat_range=LAT_EP):
    path = HINDCAST_ROOT / case / "EPflux_daily_ubar" / wave / f"EPFLUX_{wave}_{case}_members_time_plev_lat.nc"
    if not path.exists():
        raise FileNotFoundError(path)
    with xr.open_dataset(path, decode_times=False) as ds:
        ep2 = assign_member_short_coord(ds["ep2"])
        ep2_100 = ep2.sel(plev=plev_pa, method="nearest")
        ep2_100 = coslat_mean(ep2_100, lat_range=lat_range)
        # Use -ep2 so positive means upward wave activity, matching the old Fz scatter convention.
        out = (-ep2_100).load()
    return out


def ep_window_mean(ep_da: xr.DataArray, date_values, start_end=EP_WINDOW):
    start, end = start_end
    mask = date_mask(date_values, start=start, end=end)
    return ep_da.isel(lead_time=mask).mean("lead_time", skipna=True)


def load_all_wave_metrics(case=CASE, date_values=None, start_end=EP_WINDOW):
    if date_values is None:
        _, date_values = load_hindcast_o3(case)
    rows = []
    series = {}
    for wave in WAVES:
        da = load_epflux_wave(case, wave=wave)
        da = da.isel(lead_time=slice(0, len(date_values)))
        metric = ep_window_mean(da, date_values, start_end=start_end)
        series[wave] = da.assign_coords(date=("lead_time", date_values[: da.sizes["lead_time"]]))
        for member, val in zip(metric["member_short"].values, metric.values):
            rows.append({"member": str(member), "wave": wave, "ep100_upward": float(val)})
    return pd.DataFrame(rows), series


def compute_rmse_table(o3_hind: xr.DataArray, date_h, o3_ref: xr.DataArray, date_ref, start=(1, 1), end=O3_END, label="Jan01-May30"):
    mh = date_mask(date_h, start=start, end=end)
    mr = date_mask(date_ref, start=start, end=end, year=REF_YEAR)
    h = o3_hind.isel(lead_time=mh)
    r = o3_ref.isel(lead_time=mr)
    n = min(h.sizes["lead_time"], r.sizes["lead_time"])
    h = h.isel(lead_time=slice(0, n))
    r = r.isel(lead_time=slice(0, n))
    diff = h - r
    rmse = np.sqrt((diff ** 2).mean("lead_time", skipna=True))
    return pd.DataFrame({
        "member": [str(v) for v in rmse["member_short"].values],
        "RMSE_DU": rmse.values.astype(float),
        "rmse_window": label,
        "n_days": n,
    }).sort_values("RMSE_DU").reset_index(drop=True)


def merge_rmse_ep(rmse_df, ep_metric_df):
    wide = ep_metric_df.pivot(index="member", columns="wave", values="ep100_upward").reset_index()
    wide = wide.rename(columns={w: f"EP100_{w}" for w in WAVES})
    return rmse_df.merge(wide, on="member", how="left")

In [ ]:
# -----------------------------
# Product sanity check for 0008-01
# -----------------------------
expected = {
    "partial_O3": CASE_ROOT / "partial_O3" / f"{CASE}_partial_O3_all_ranges_members.nc",
    "EP all": CASE_ROOT / "EPflux_daily_ubar" / "all_waves" / f"EPFLUX_all_waves_{CASE}_members_time_plev_lat.nc",
    "EP wave1": CASE_ROOT / "EPflux_daily_ubar" / "wave1" / f"EPFLUX_wave1_{CASE}_members_time_plev_lat.nc",
    "EP wave2": CASE_ROOT / "EPflux_daily_ubar" / "wave2" / f"EPFLUX_wave2_{CASE}_members_time_plev_lat.nc",
    "EP rest": CASE_ROOT / "EPflux_daily_ubar" / "wave_rest" / f"EPFLUX_wave_rest_{CASE}_members_time_plev_lat.nc",
    "FWD": CASE_ROOT / "final_warming_date" / f"{CASE}_FWD_plev_member.nc",
    "AO/NAM projection": CASE_ROOT / "NAM_B2000WCN_projection" / f"{CASE}_AO_NAM_B2000WCN_projection_members.nc",
}
for name, path in expected.items():
    print(f"{name:18s}: {path.exists()}  {path}")

with xr.open_dataset(expected["partial_O3"], decode_times=False) as ds:
    print("\npartial_O3 dims:", dict(ds.sizes))
    print("partial_O3 vars:", list(ds.data_vars))
with xr.open_dataset(expected["EP all"], decode_times=False) as ds:
    print("\nEP all dims:", dict(ds.sizes))
    print("EP attrs:", {k: ds.attrs.get(k) for k in ["method", "do_ubar", "use_omega_w_correction"]})

## 图：O3 RMSE 与早期 EPFlux

**做什么**：复现 cleaned Hindcast 数据中的 O3 RMSE vs 早期波活动散点图。

**怎么做**：O3 RMSE 使用 `60-90N, 30-70 hPa` partial O3，窗口为 Jan1-May30，对比 BWCN 第 0008 年。EPFlux 指标为 `EP100 = mean(-ep2)`，即 100 hPa、40-80N、Jan20-Feb10 平均的垂直 EP flux 分量；不是 divergence。

**科学问题**：早期进入平流层的波活动强弱，是否能解释后续臭氧柱偏离参考年的程度？

**预期**：如果波活动控制了可预报性，EP100 越强的成员应有更大的 O3 RMSE。

**运行后解读**：待图生成后填写。


In [ ]:
# -----------------------------
# Reproduce O3 RMSE vs winter wave activity with cleaned data
# -----------------------------
o3_h, date_h = load_hindcast_o3(CASE)
o3_ref, date_ref = load_bwcn_ref_o3(REF_YEAR)

ep_metric_df, ep_series = load_all_wave_metrics(CASE, date_h, EP_WINDOW)
rmse_all = compute_rmse_table(o3_h, date_h, o3_ref, date_ref, start=(1, 1), end=O3_END, label="Jan01-May30")
rmse_ep_all = merge_rmse_ep(rmse_all, ep_metric_df)
rmse_ep_all.to_csv(TABLE_DIR / f"{CASE}_rmse_epflux_wave_metrics_Jan20_Feb10.csv", index=False)

print(rmse_ep_all.head())
print("\nRMSE range:", rmse_ep_all["RMSE_DU"].min(), rmse_ep_all["RMSE_DU"].max())
print("Saved table:", TABLE_DIR / f"{CASE}_rmse_epflux_wave_metrics_Jan20_Feb10.csv")

fig, ax = plt.subplots(figsize=(7.4, 5.4), constrained_layout=True)
scatter_fit(
    ax,
    rmse_ep_all,
    "EP100_all_waves",
    "RMSE_DU",
    title="Cleaned data: O3 RMSE vs early winter all-wave EP flux",
    xlabel="Mean 100 hPa -EP2, 40-80N, Jan20-Feb10",
    ylabel="O3 RMSE, 60-90N 30-70 hPa, Jan1-May30 (DU)",
    color="dodgerblue",
)
savefig(fig, f"{CASE}_reproduce_RMSE_vs_allwave_EPFlux")
plt.show()

## 图：EPFlux 分波贡献

**做什么**：比较 all waves、wave1、wave2、wave rest、wave1+wave2 与 O3 RMSE 的关系。

**怎么做**：所有 EP 指标均为 `mean(-ep2)`，100 hPa、40-80N、Jan20-Feb10；这里比较的是垂直 EP flux 分量，不是 divergence。

**科学问题**：可预报性信号主要来自行星波 1、波 2、synoptic/rest，还是 all-wave 的综合效应？

**预期**：如果长波是主要来源，wave1+wave2 应接近 all waves；如果 synoptic/rest 重要，rest 或 all waves 会明显更强。

**运行后解读**：待图生成后填写。


In [ ]:
# -----------------------------
# Which EP-flux wave component matters most?
# EP100_* below is a scalar member metric: mean -ep2 at 100 hPa, 40-80N, Jan20-Feb10.
# It is the vertical EP-flux component with sign flipped so positive follows the old upward-Fz convention;
# it is not div1/div2 or EP-flux divergence.
# -----------------------------
rmse_ep_all = rmse_ep_all.copy()
rmse_ep_all["EP100_wave1_plus_wave2"] = rmse_ep_all["EP100_wave1"] + rmse_ep_all["EP100_wave2"]

EP_SCALAR_COLS = [
    ("EP100_all_waves", "all_waves", "All", "black"),
    ("EP100_wave1", "wave1", "W1", "tab:blue"),
    ("EP100_wave2", "wave2", "W2", "tab:orange"),
    ("EP100_wave_rest", "wave_rest", "Rest", "tab:green"),
    ("EP100_wave1_plus_wave2", "wave1_plus_wave2", "W1+W2", "tab:purple"),
]

summary_rows = []
for col, key, label, _ in EP_SCALAR_COLS:
    r, p = finite_corr(rmse_ep_all[col], rmse_ep_all["RMSE_DU"])
    summary_rows.append({"metric": key, "label": label, "R_vs_RMSE": r, "P": p, "definition": EP_SCALAR_DESCRIPTION})
r_all_combo, p_all_combo = finite_corr(rmse_ep_all["EP100_wave1_plus_wave2"], rmse_ep_all["EP100_all_waves"])
summary_rows.append({
    "metric": "all_waves_vs_wave1_plus_wave2",
    "label": "All vs W1+W2",
    "R_vs_RMSE": np.nan,
    "P": np.nan,
    "R_aux": r_all_combo,
    "P_aux": p_all_combo,
    "definition": EP_SCALAR_DESCRIPTION,
})
wave_corr = pd.DataFrame(summary_rows)
wave_corr.to_csv(TABLE_DIR / f"{CASE}_wave_component_correlations.csv", index=False)
print(wave_corr)

fig, axes = plt.subplots(2, 3, figsize=(15.8, 8.8), constrained_layout=True)
plot_specs = [
    ("EP100_all_waves", "RMSE_DU", "All vs RMSE", "EP100 all", "O3 RMSE (DU)", "black"),
    ("EP100_wave1", "RMSE_DU", "W1 vs RMSE", "EP100 W1", "O3 RMSE (DU)", "tab:blue"),
    ("EP100_wave2", "RMSE_DU", "W2 vs RMSE", "EP100 W2", "O3 RMSE (DU)", "tab:orange"),
    ("EP100_wave_rest", "RMSE_DU", "Rest vs RMSE", "EP100 rest", "O3 RMSE (DU)", "tab:green"),
    ("EP100_wave1_plus_wave2", "RMSE_DU", "W1+W2 vs RMSE", "EP100 W1+W2", "O3 RMSE (DU)", "tab:purple"),
    ("EP100_wave1_plus_wave2", "EP100_all_waves", "All vs W1+W2", "EP100 W1+W2", "EP100 all", "0.25"),
]
for ax, (xcol, ycol, title, xlabel, ylabel, color) in zip(axes.ravel(), plot_specs):
    scatter_fit(ax, rmse_ep_all, xcol, ycol, title=title, xlabel=xlabel, ylabel=ylabel, color=color, annotate_members=False)
fig.suptitle("EP100 = mean(-ep2), 100 hPa, 40-80N, Jan20-Feb10", fontsize=11)
savefig(fig, f"{CASE}_RMSE_vs_EPFlux_by_wave")
plt.show()


## 图：spread timing：O3、EPFlux 与 U60N10

**做什么**：单独检查 ensemble spread 出现的时间顺序，比较 O3、各分波 EPFlux、以及 60N 10 hPa zonal-mean wind 的 spread 是否同步或谁领先。

**怎么做**：O3 使用 `60-90N, 30-70 hPa` partial O3；EPFlux 使用 `EP100 = mean(-ep2)`，即 100 hPa、40-80N；U60N10 从 raw U 插值到 10 hPa、取 60N 最近纬度并做 zonal mean。所有曲线都画相对 lead day 0 的 spread change，并按各自时间序列标准化，因此曲线从 0 开始，只比较时序和相对增长，不比较物理量单位大小。

**科学问题**：EPFlux spread 是否早于 O3 spread？极夜急流 spread 是否与 EPFlux spread 同步，提示动力过程先于臭氧响应？

**预期**：如果早期波活动是臭氧可预报性差异的源头，EPFlux spread 应早于或至少不晚于 O3 spread；U60N10 spread 可能在波活动增强后随之扩大。

**运行后解读**：待图生成后填写。


In [ ]:
# -----------------------------
# Spread timing: does EP-flux spread lead ozone and vortex-wind spread?
# O3: cleaned partial_O3, 60-90N, 30-70 hPa.
# EP100 components: -ep2 at 100 hPa, 40-80N.
# U60N10: raw U interpolated to 10 hPa, nearest 60N, zonal mean.
# All curves are spread changes relative to lead day 0 and standardized separately.
# -----------------------------

def zero_start_standardized(values):
    arr = np.asarray(values, dtype=float)
    if arr.size == 0 or not np.isfinite(arr[0]):
        return arr * np.nan
    delta = arr - arr[0]
    scale = np.nanstd(delta[1:])
    if not np.isfinite(scale) or scale == 0:
        scale = 1.0
    return delta / scale


def interp_logp_single_level(da_var: xr.DataArray, p_hyb: xr.DataArray, p_tgt_pa: float) -> xr.DataArray:
    target = np.array([float(p_tgt_pa)], dtype=float)

    def _interp_col(vcol, pcol):
        vcol = np.asarray(vcol, dtype=float)
        pcol = np.asarray(pcol, dtype=float)
        mask = np.isfinite(vcol) & np.isfinite(pcol) & (pcol > 0)
        if mask.sum() < 2:
            return np.array([np.nan], dtype=float)
        x = np.log(pcol[mask])
        y = vcol[mask]
        order = np.argsort(x)
        return np.interp(np.log(target), x[order], y[order], left=np.nan, right=np.nan)

    out = xr.apply_ufunc(
        _interp_col,
        da_var,
        p_hyb,
        input_core_dims=[["lev"], ["lev"]],
        output_core_dims=[["plev"]],
        vectorize=True,
        dask="parallelized",
        output_dtypes=[float],
        dask_gufunc_kwargs={"output_sizes": {"plev": 1}},
    )
    return out.assign_coords(plev=target).isel(plev=0, drop=True)


def u60n10_from_file_for_spread(path: Path, start_end=((1, 1), (5, 30))) -> Tuple[xr.DataArray, np.ndarray]:
    with xr.open_dataset(path, decode_times=False) as ds:
        date = np.asarray(ds["date"].values, dtype=np.int64)
        mask = date_mask(date, start=start_end[0], end=start_end[1])
        ds = ds.isel(time=mask).sel(lat=60.0, method="nearest")
        date = date[mask]
        p_mid = ds["hyam"] * ds["P0"] + ds["hybm"] * ds["PS"]
        u = interp_logp_single_level(
            ds["U"].transpose("time", "lon", "lev"),
            p_mid.transpose("time", "lon", "lev"),
            PLEV_U_PA,
        )
        u_zm = u.mean("lon", skipna=True).load()
    u_zm = u_zm.rename({"time": "lead_time"}).assign_coords(lead_time=np.arange(len(date)), date=("lead_time", date))
    return u_zm, date


def build_u60n10_case_cache_for_spread(case=CASE, overwrite=False):
    out = CACHE_DIR / f"U60N10_{case}_members.nc"
    if out.exists() and not overwrite:
        da = xr.open_dataarray(out).load()
        date = np.asarray(da["date"].values, dtype=np.int64)
        return da, date
    files = sorted((HINDCAST_ROOT / case / "U").glob("*.U.nc"))
    if not files:
        raise FileNotFoundError(f"No U files for {case}")
    das, mids = [], []
    date0 = None
    for f in files:
        mid = member_short_id(f.name)
        print("U60N10 spread cache", case, mid)
        da, date = u60n10_from_file_for_spread(f)
        das.append(da)
        mids.append(mid)
        date0 = date
    out_da = xr.concat(das, dim=pd.Index(mids, name="member"))
    out_da.name = "U60N10"
    out_da.to_netcdf(out)
    print(f"Saved cache: {out}")
    return out_da, date0

x = np.arange(len(date_h))
fig, ax = plt.subplots(figsize=(11.8, 5.5), constrained_layout=True)

o3_spread = o3_h.std("member", skipna=True).values
ax.plot(x, zero_start_standardized(o3_spread), color="crimson", lw=2.6, label="O3 partial column")

spread_sources = {wave: ep_series[wave].isel(lead_time=slice(0, len(date_h))) for wave in WAVES}
spread_sources["wave1_plus_wave2"] = spread_sources["wave1"] + spread_sources["wave2"]
for wave, da in spread_sources.items():
    spread = da.std("member", skipna=True).values
    ax.plot(x, zero_start_standardized(spread), lw=1.7, label=f"EP100 {WAVE_LABELS[wave]}")

try:
    u60n10_h, u60n10_date = build_u60n10_case_cache_for_spread(CASE)
    n = min(len(date_h), u60n10_h.sizes["lead_time"])
    u_spread = u60n10_h.isel(lead_time=slice(0, n)).std("member", skipna=True).values
    ax.plot(np.arange(n), zero_start_standardized(u_spread), color="0.15", ls="--", lw=2.3, label="U60N10")
except Exception as exc:
    print(f"Skip U60N10 spread curve: {exc}")

mask_ep = date_mask(date_h, start=EP_WINDOW[0], end=EP_WINDOW[1])
if mask_ep.any():
    ax.axvspan(np.where(mask_ep)[0][0], np.where(mask_ep)[0][-1], color="0.85", alpha=0.5, label="EP window")
positions, labels = month_ticks(date_h)
ax.set_xticks(positions)
ax.set_xticklabels(labels)
ax.axhline(0, color="0.35", lw=0.8)
ax.set_ylabel("Spread change from day 0 (standardized)")
ax.set_title("Spread timing: O3, EP100 wave components, and U60N10")
ax.legend(ncol=3, fontsize=8.8)
savefig(fig, f"{CASE}_spread_timing_O3_EPFlux_waves")
plt.show()


## 图：RMSE 时间窗口敏感性

**做什么**：保留原测试，比较 Jan1、Jan20、Feb20 开始计算 O3 RMSE 时，RMSE 与 all-wave EP100 的关系是否稳定。

**怎么做**：EPFlux 固定为 100 hPa、40-80N、Jan20-Feb10 的 all-wave `-ep2`；仅改变 O3 RMSE 的计算起始日期。

**科学问题**：EPFlux-RMSE 关系是否只是由初期 O3 偏差造成，还是对后期误差也有解释力？

**预期**：如果关系稳健，去掉早期 O3 后相关仍应存在。

**运行后解读**：待图生成后填写。


In [ ]:
# -----------------------------
# RMSE-window sensitivity: whole window vs Jan20+ vs Feb20+
# -----------------------------
rmse_windows = [
    ("RMSE_Jan01_May30", (1, 1), "Jan1-May30"),
    ("RMSE_Jan20_May30", (1, 20), "Jan20-May30"),
    ("RMSE_Feb20_May30", (2, 20), "Feb20-May30"),
]
merged_frames = []
for col, start, label in rmse_windows:
    df = compute_rmse_table(o3_h, date_h, o3_ref, date_ref, start=start, end=O3_END, label=label)
    df = df.rename(columns={"RMSE_DU": col})[["member", col, "n_days"]]
    merged_frames.append(df)
rmse_multi = merged_frames[0]
for df in merged_frames[1:]:
    rmse_multi = rmse_multi.merge(df, on="member", how="inner", suffixes=("", "_y"))
rmse_multi = rmse_multi.merge(rmse_ep_all[["member"] + [f"EP100_{w}" for w in WAVES]], on="member", how="left")
rmse_multi.to_csv(TABLE_DIR / f"{CASE}_rmse_window_sensitivity.csv", index=False)
print(rmse_multi.head())

fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.9), constrained_layout=True, sharex=True)
for ax, (col, _, label) in zip(axes, rmse_windows):
    dfp = rmse_multi.rename(columns={col: "RMSE_window"})
    scatter_fit(
        ax, dfp, "EP100_all_waves", "RMSE_window",
        title=label,
        xlabel="Mean 100 hPa all-wave -EP2, Jan20-Feb10",
        ylabel="O3 RMSE (DU)",
        color="tab:purple",
        annotate_members=False,
    )
savefig(fig, f"{CASE}_RMSE_window_sensitivity_vs_allwave_EPFlux")
plt.show()

## 图：AO 与 EPFlux/O3 RMSE

**做什么**：检查 B2000WCN 第一模态投影得到的 Hindcast AO 是否与 O3 RMSE 或同窗口 EPFlux 有关。

**怎么做**：AO 使用 cleaned `NAM_B2000WCN_projection`，即 1000 hPa zonal-mean AO 投影到 B2000WCN 第一模态。基础图使用 `Jan20-Feb10`，让 `AO vs O3 RMSE` 和 `AO vs concurrent all-wave EP` 的 AO 时间窗口一致。随后扩展到 `Jan`、`Jan-Feb`、`Jan-Mar`；每个扩展窗口都会重新计算同窗口的 EP100 分波指标。

**科学问题**：annular mode 状态是否能解释早期/累积波活动异常，进而解释后续臭氧可预报性差异？

**预期**：如果 AO 是波活动异常的低维源头，AO 与同窗口 EP100 或 O3 RMSE 应有系统相关，并可能随窗口扩展增强或减弱。

**运行后解读**：待图生成后填写。


In [ ]:
# -----------------------------
# AO test: use B2000WCN-projected AO/NAM if available.
# -----------------------------

def load_projected_ao(case=CASE):
    new_path = HINDCAST_ROOT / case / "NAM_B2000WCN_projection" / f"{case}_AO_NAM_B2000WCN_projection_members.nc"
    if not new_path.exists():
        print(f"Missing projected AO/NAM product: {new_path}")
        print("The legacy NAM/*.nc file lacks a reliable member axis for this scatter, so AO tests are skipped until the projected product is generated.")
        return None, None
    with xr.open_dataset(new_path, decode_times=False) as ds:
        ao = assign_member_short_coord(ds["AO_Index"]).load()
        date = one_dim_date(ds)
    if int(ao.count()) == 0:
        print("Projected AO file exists, but AO_Index is all NaN. Regenerate it with the fixed script:")
        print("/home/weiji/miniconda3/envs/jimnew/bin/python Hindcast_experiment/date_treatment/scripts/compute_hindcast_ao_nam_b2000wcn_projection.py --cases 0008-01 --overwrite --max-workers 4")
        return None, None
    ao = ao.assign_coords(date=("lead_time", date))
    return ao, date


def window_tag(label: str) -> str:
    return label.replace("+", "plus").replace("-", "_").replace(" ", "_")


def ao_mean_for_window(ao_da: xr.DataArray, date_values: np.ndarray, start_end) -> xr.DataArray:
    mask = date_mask(date_values, start=start_end[0], end=start_end[1])
    return ao_da.isel(lead_time=mask).mean("lead_time", skipna=True)


def ep_metrics_for_window(ep_series_dict: Dict[str, xr.DataArray], date_values: np.ndarray, start_end) -> pd.DataFrame:
    rows = []
    for wave, da in ep_series_dict.items():
        metric = ep_window_mean(da, date_values, start_end=start_end)
        for member, val in zip(metric["member_short"].values, metric.values):
            rows.append({"member": str(member), "wave": wave, "ep100_upward": float(val)})
    wide = pd.DataFrame(rows).pivot(index="member", columns="wave", values="ep100_upward").reset_index()
    wide = wide.rename(columns={w: f"EP100_{w}" for w in WAVES})
    wide["EP100_wave1_plus_wave2"] = wide["EP100_wave1"] + wide["EP100_wave2"]
    return wide


def build_ao_ep_window_table(ao_da: xr.DataArray, date_values: np.ndarray, label: str, start_end) -> pd.DataFrame:
    ao_metric = ao_mean_for_window(ao_da, date_values, start_end)
    rows = [
        {"member": str(mid), "AO_mean": float(ao_metric.isel(member=i)), "AO_window": label}
        for i, mid in enumerate(ao_da["member_short"].values)
    ]
    ao_df = pd.DataFrame(rows)
    ep_df = ep_metrics_for_window(ep_series, date_h, start_end)
    out = rmse_all.merge(ao_df, on="member", how="left").merge(ep_df, on="member", how="left")
    return out


ao, date_ao = load_projected_ao(CASE)
if ao is not None:
    ao_window_tables = []
    for label, start_end in AO_TEST_WINDOWS.items():
        df = build_ao_ep_window_table(ao, date_ao, label, start_end)
        ao_window_tables.append(df)
    ao_all_windows = pd.concat(ao_window_tables, ignore_index=True)
    ao_all_windows.to_csv(TABLE_DIR / f"{CASE}_AO_EP_RMSE_metrics_by_window.csv", index=False)

    # Backward-compatible base table/figures now use Jan20-Feb10 AO for every AO panel.
    ao_join = ao_all_windows.loc[ao_all_windows["AO_window"] == AO_BASE_WINDOW_LABEL].copy()
    ao_join.to_csv(TABLE_DIR / f"{CASE}_AO_EP_RMSE_metrics.csv", index=False)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True)
    scatter_fit(
        axes[0], ao_join, "AO_mean", "RMSE_DU",
        "AO vs O3 RMSE", f"AO mean {AO_BASE_WINDOW_LABEL}", "O3 RMSE (DU)", "tab:cyan",
    )
    scatter_fit(
        axes[1], ao_join, "AO_mean", "EP100_all_waves",
        "AO vs concurrent all-wave EP", f"AO mean {AO_BASE_WINDOW_LABEL}",
        f"100 hPa all-wave -EP2 ({AO_BASE_WINDOW_LABEL})", "tab:cyan",
    )
    fig.suptitle(f"AO diagnostics, {AO_BASE_WINDOW_LABEL}", fontsize=12)
    savefig(fig, f"{CASE}_AO_tests_RMSE_EPFlux")
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True)
    for ax, label in zip(axes, ["Jan", "Jan-Feb"]):
        df = ao_all_windows.loc[ao_all_windows["AO_window"] == label]
        scatter_fit(
            ax, df, "AO_mean", "RMSE_DU",
            f"{label}: AO vs O3 RMSE", f"AO mean {label}", "O3 RMSE (DU)", "tab:cyan",
        )
    savefig(fig, f"{CASE}_AO_vs_RMSE_Jan_JanFeb")
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True)
    for ax, label in zip(axes, ["Jan-Mar", "Jan20-Feb10"]):
        df = ao_all_windows.loc[ao_all_windows["AO_window"] == label]
        scatter_fit(
            ax, df, "AO_mean", "RMSE_DU",
            f"{label}: AO vs O3 RMSE", f"AO mean {label}", "O3 RMSE (DU)", "tab:cyan",
        )
    savefig(fig, f"{CASE}_AO_vs_RMSE_JanMar_Jan20Feb10")
    plt.show()

    fig, axes = plt.subplots(len(AO_TEST_WINDOWS), 2, figsize=(12.5, 14.2), constrained_layout=True)
    for r, (label, _) in enumerate(AO_TEST_WINDOWS.items()):
        df = ao_all_windows.loc[ao_all_windows["AO_window"] == label]
        scatter_fit(
            axes[r, 0], df, "AO_mean", "RMSE_DU",
            f"{label}: AO vs O3 RMSE", f"AO mean {label}", "O3 RMSE (DU)", "tab:cyan",
            annotate_members=False,
        )
        scatter_fit(
            axes[r, 1], df, "AO_mean", "EP100_all_waves",
            f"{label}: AO vs all-wave EP", f"AO mean {label}", f"EP100 all ({label})", "tab:cyan",
            annotate_members=False,
        )
    fig.suptitle("AO window sensitivity: AO, O3 RMSE, and concurrent all-wave EP100", fontsize=12)
    savefig(fig, f"{CASE}_AO_tests_RMSE_EPFlux_windows")
    plt.show()

    fig, axes = plt.subplots(2, 2, figsize=(12, 9), constrained_layout=True)
    for ax, wave in zip(axes.ravel(), WAVES):
        scatter_fit(
            ax, ao_join, "AO_mean", f"EP100_{wave}",
            f"AO vs {WAVE_LABELS[wave]}", f"AO mean {AO_BASE_WINDOW_LABEL}",
            f"{WAVE_LABELS[wave]} -EP2 ({AO_BASE_WINDOW_LABEL})", "tab:cyan", annotate_members=False,
        )
    fig.suptitle(f"AO vs EP100 wave components, {AO_BASE_WINDOW_LABEL}", fontsize=12)
    savefig(fig, f"{CASE}_AO_vs_wave_components")
    plt.show()

    wave_cols = [
        ("EP100_all_waves", "All"),
        ("EP100_wave1", "W1"),
        ("EP100_wave2", "W2"),
        ("EP100_wave_rest", "Rest"),
        ("EP100_wave1_plus_wave2", "W1+W2"),
    ]
    fig, axes = plt.subplots(len(AO_TEST_WINDOWS), len(wave_cols), figsize=(18.5, 14.2), constrained_layout=True)
    for r, (label, _) in enumerate(AO_TEST_WINDOWS.items()):
        df = ao_all_windows.loc[ao_all_windows["AO_window"] == label]
        for c, (col, short_label) in enumerate(wave_cols):
            scatter_fit(
                axes[r, c], df, "AO_mean", col,
                f"{label}: AO vs {short_label}", f"AO mean {label}", f"EP100 {short_label}",
                "tab:cyan", annotate_members=False,
            )
    fig.suptitle("AO window sensitivity: concurrent EP100 wave components", fontsize=12)
    savefig(fig, f"{CASE}_AO_vs_wave_components_windows")
    plt.show()

    # Also split the 4 x 5 window-sensitivity panel into three 2 x 2 figures.
    # Each figure shows the four AO windows: Jan20-Feb10, Jan, Jan-Feb, and Jan-Mar.
    def plot_ao_metric_windows_2x2(metric_col: str, metric_label: str, out_suffix: str):
        fig, axes = plt.subplots(2, 2, figsize=(11.5, 9.0), constrained_layout=True)
        for ax, (label, _) in zip(axes.ravel(), AO_TEST_WINDOWS.items()):
            df = ao_all_windows.loc[ao_all_windows["AO_window"] == label]
            scatter_fit(
                ax, df, "AO_mean", metric_col,
                f"{label}: AO vs {metric_label}", f"AO mean {label}", f"EP100 {metric_label} ({label})",
                "tab:cyan", annotate_members=False,
            )
        fig.suptitle(f"AO window sensitivity: AO vs EP100 {metric_label}", fontsize=12)
        savefig(fig, f"{CASE}_AO_vs_{out_suffix}_windows")
        plt.show()

    plot_ao_metric_windows_2x2("EP100_all_waves", "All waves", "all_waves")
    plot_ao_metric_windows_2x2("EP100_wave1_plus_wave2", "W1+W2", "wave1_plus_wave2")
    plot_ao_metric_windows_2x2("EP100_wave_rest", "Rest", "wave_rest")


## 图：AO 领先、EPFlux 滞后的异步窗口测试

**做什么**：测试 AO 是否先于进入平流层的 wave activity。固定测试采用 30 天 AO 窗口和 30 天 EPFlux 窗口，EPFlux 窗口相对 AO 晚 10 天；同时扫描不同 AO 起始日、window 长度和 delay，寻找 member-to-member 相关最强的时间组合。

**怎么做**：AO 指标是 B2000WCN-projected AO 的成员平均值；EPFlux 指标仍为 `EP100 = mean(-ep2)`，即 100 hPa、40-80N cos-lat 平均的垂直 EP-flux 分量，不是 divergence。固定测试锚定原先的 Jan20 EPFlux 起始日，因此 AO = Jan10-Feb08，EPFlux = Jan20-Feb18。扫描只考虑 EPFlux 晚于 AO 的正 delay。

**科学问题**：如果早期 AO 调制后续 planetary-wave 入射，那么 AO 窗口应当与稍晚的 all-wave、wave1+wave2 或 rest EP100 有显著相关；若只有同步相关而无滞后相关，AO 更可能只是同期环流状态的伴随指标。

**预期**：优先关注 `|R| >= 0.5` 且 `p < 0.05` 的组合；若固定 10 天 lag 或扫描最佳组合都不显著，就不把 AO lead 作为主要 source 解释。

**运行后解读**：当前 0008-01 运行结果显示，固定 30 天窗口、AO 领先 10 天的测试不显著：all-wave `R≈0.20, p≈0.30`，W1+W2 `R≈0.15, p≈0.42`。探索扫描里显著点主要集中在 20 天窗口，尤其早 1 月 AO vs 稍晚 EPFlux 的 wave2/W1+W2 负相关和 rest 正相关；因此额外输出 20 天 heatmap、20 天窗口/5 天 lag 的相关性折线图，以及 AO Jan01-Jan20 vs EP Jan06-Jan25 的 all-wave/W1+W2 散点图。指定窗口读数为：All `R=-0.622, p=2.45e-4`，W1+W2 `R=-0.699, p=1.7e-5`，wave2 `R=-0.761, p=1.04e-6`，rest `R=0.693, p=2.2e-5`。若只坚持 30 天、10 天 lag，目前不应把 AO lead 作为强 source 结论。

In [ ]:
# -----------------------------
# AO lead / delayed EPFlux tests
# -----------------------------
# Fixed hypothesis: AO is averaged over a 30-day window, EPFlux is averaged over
# the following 30-day window delayed by +10 days. The fixed test anchors the
# EPFlux start date at Jan20, so AO = Jan10-Feb08 and EPFlux = Jan20-Feb18.
# The scan then varies AO start date, window length, and delay to see whether a
# stronger, physically coherent lag emerges.

if "date_h" not in globals() or "ep_series" not in globals():
    o3_h, date_h = load_hindcast_o3(CASE)
    _, ep_series = load_all_wave_metrics(CASE, date_h, EP_WINDOW)

if "load_projected_ao" not in globals():
    def load_projected_ao(case=CASE):
        new_path = HINDCAST_ROOT / case / "NAM_B2000WCN_projection" / f"{case}_AO_NAM_B2000WCN_projection_members.nc"
        if not new_path.exists():
            print(f"Missing projected AO/NAM product: {new_path}")
            return None, None
        with xr.open_dataset(new_path, decode_times=False) as ds:
            ao = assign_member_short_coord(ds["AO_Index"]).load()
            date = one_dim_date(ds)
        if int(ao.count()) == 0:
            print("Projected AO file exists, but AO_Index is all NaN. Regenerate Hindcast AO/NAM first.")
            return None, None
        return ao.assign_coords(date=("lead_time", date)), date

if "ao" not in globals() or "date_ao" not in globals():
    ao, date_ao = load_projected_ao(CASE)

_DOY_MONTH_START = np.array([0, 31, 59, 90, 120, 151, 181, 212, 243, 273, 304, 334], dtype=int)
_MONTH_ABBR = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]


def mmdd_to_doy(month: int, day: int) -> int:
    return int(_DOY_MONTH_START[int(month) - 1] + int(day))


def date_to_doy(date_values) -> np.ndarray:
    _, month, day = date_parts(date_values)
    return _DOY_MONTH_START[month.astype(int) - 1] + day.astype(int)


def doy_to_mmdd(doy: int) -> Tuple[int, int]:
    doy = int(doy)
    month = int(np.searchsorted(_DOY_MONTH_START + 1, doy, side="right"))
    month = min(max(month, 1), 12)
    day = doy - int(_DOY_MONTH_START[month - 1])
    return month, day


def doy_window_label(start_doy: int, window_days: int) -> str:
    sm, sd = doy_to_mmdd(start_doy)
    em, ed = doy_to_mmdd(start_doy + window_days - 1)
    return f"{_MONTH_ABBR[sm - 1]}{sd:02d}-{_MONTH_ABBR[em - 1]}{ed:02d}"


def doy_window_mask(date_values, start_doy: int, window_days: int) -> np.ndarray:
    doy = date_to_doy(date_values)
    end_doy = int(start_doy) + int(window_days) - 1
    return (doy >= int(start_doy)) & (doy <= end_doy)


def member_mean_for_doy_window(da: xr.DataArray, date_values, start_doy: int, window_days: int) -> xr.DataArray:
    n = min(int(da.sizes["lead_time"]), len(date_values))
    mask = doy_window_mask(np.asarray(date_values)[:n], start_doy, window_days)
    if mask.sum() == 0:
        return da.isel(lead_time=slice(0, 0)).mean("lead_time", skipna=True)
    return da.isel(lead_time=np.where(mask)[0]).mean("lead_time", skipna=True)


def ao_ep_async_table(
    ao_da: xr.DataArray,
    ao_date_values,
    ep_series_dict: Dict[str, xr.DataArray],
    ep_date_values,
    ao_start_doy: int,
    window_days: int,
    delay_days: int,
) -> pd.DataFrame:
    ep_start_doy = int(ao_start_doy) + int(delay_days)
    ao_metric = member_mean_for_doy_window(ao_da, ao_date_values, ao_start_doy, window_days)
    ao_df = pd.DataFrame({
        "member": [str(mid) for mid in ao_metric["member_short"].values],
        "AO_mean": ao_metric.values.astype(float),
    })

    ep_wide = None
    for wave in WAVES:
        metric = member_mean_for_doy_window(ep_series_dict[wave], ep_date_values, ep_start_doy, window_days)
        this = pd.DataFrame({
            "member": [str(mid) for mid in metric["member_short"].values],
            f"EP100_{wave}": metric.values.astype(float),
        })
        ep_wide = this if ep_wide is None else ep_wide.merge(this, on="member", how="outer")
    ep_wide["EP100_wave1_plus_wave2"] = ep_wide["EP100_wave1"] + ep_wide["EP100_wave2"]

    out = ao_df.merge(ep_wide, on="member", how="inner")
    out["ao_start_doy"] = int(ao_start_doy)
    out["ep_start_doy"] = int(ep_start_doy)
    out["window_days"] = int(window_days)
    out["delay_days"] = int(delay_days)
    out["AO_window"] = doy_window_label(ao_start_doy, window_days)
    out["EP_window"] = doy_window_label(ep_start_doy, window_days)
    return out


ASYNC_WAVE_COLS = [
    ("EP100_all_waves", "All waves", "black"),
    ("EP100_wave1", "Wave 1", "tab:blue"),
    ("EP100_wave2", "Wave 2", "tab:orange"),
    ("EP100_wave_rest", "Rest", "tab:green"),
    ("EP100_wave1_plus_wave2", "W1+W2", "tab:purple"),
]


def ao_ep_corr_rows(df: pd.DataFrame) -> list[dict]:
    rows = []
    for col, label, _ in ASYNC_WAVE_COLS:
        r, p = finite_corr(df["AO_mean"], df[col])
        rows.append({
            "metric": col.replace("EP100_", ""),
            "label": label,
            "R_AO_vs_EP100": r,
            "P": p,
            "ao_start_doy": int(df["ao_start_doy"].iloc[0]),
            "ep_start_doy": int(df["ep_start_doy"].iloc[0]),
            "window_days": int(df["window_days"].iloc[0]),
            "delay_days": int(df["delay_days"].iloc[0]),
            "AO_window": str(df["AO_window"].iloc[0]),
            "EP_window": str(df["EP_window"].iloc[0]),
            "n_members": int(df[["AO_mean", col]].dropna().shape[0]),
            "definition": "AO mean leads EP100; EP100 is mean(-ep2), 100 hPa, 40-80N, not divergence.",
        })
    return rows


if ao is None:
    print("Skip asynchronous AO/EPFlux test: projected AO is unavailable.")
else:
    ASYNC_WINDOW_DAYS = 30
    ASYNC_DELAY_DAYS = 10
    ASYNC_EP_ANCHOR_START = (1, 20)
    ASYNC_EP_ANCHOR_DOY = mmdd_to_doy(*ASYNC_EP_ANCHOR_START)
    ASYNC_AO_START_DOY = ASYNC_EP_ANCHOR_DOY - ASYNC_DELAY_DAYS

    fixed_async = ao_ep_async_table(
        ao, date_ao, ep_series, date_h,
        ao_start_doy=ASYNC_AO_START_DOY,
        window_days=ASYNC_WINDOW_DAYS,
        delay_days=ASYNC_DELAY_DAYS,
    )
    fixed_async.to_csv(TABLE_DIR / f"{CASE}_AO_leads_EPFlux_lag10_30day_metrics.csv", index=False)
    fixed_corr = pd.DataFrame(ao_ep_corr_rows(fixed_async))
    fixed_corr.to_csv(TABLE_DIR / f"{CASE}_AO_leads_EPFlux_lag10_30day_correlations.csv", index=False)
    print("Fixed asynchronous test: AO leads EPFlux by 10 days, 30-day windows")
    print(fixed_corr[["label", "R_AO_vs_EP100", "P", "AO_window", "EP_window", "n_members"]])

    fig, axes = plt.subplots(2, 3, figsize=(15.6, 8.8), constrained_layout=True)
    for ax, (col, label, color) in zip(axes.ravel()[:5], ASYNC_WAVE_COLS):
        scatter_fit(
            ax, fixed_async, "AO_mean", col,
            f"AO lead 10d vs {label}",
            f"AO mean {fixed_async['AO_window'].iloc[0]}",
            f"EP100 {label} {fixed_async['EP_window'].iloc[0]}",
            color=color,
            annotate_members=False,
        )
    axes.ravel()[5].axis("off")
    axes.ravel()[5].text(
        0.0, 0.95,
        "Fixed hypothesis\n"
        f"AO: {fixed_async['AO_window'].iloc[0]}\n"
        f"EPFlux: {fixed_async['EP_window'].iloc[0]}\n"
        "Window: 30 d\nDelay: +10 d\n"
        "EP100 = mean(-ep2), 100 hPa, 40-80N\n"
        "Positive EP100 follows the upward-Fz convention.",
        va="top", ha="left", fontsize=10,
    )
    fig.suptitle("Asynchronous AO -> EPFlux test: AO leads wave activity by 10 days", fontsize=12)
    savefig(fig, f"{CASE}_AO_leads_EPFlux_lag10_30day")
    plt.show()

    scan_windows = [20, 30, 40]
    scan_delays = [0, 5, 10, 15, 20, 30, 40]
    scan_start_step = 5
    max_ep_doy = int(np.nanmax(date_to_doy(date_h)))
    max_ao_doy = int(np.nanmax(date_to_doy(date_ao)))
    scan_rows = []
    for window_days in scan_windows:
        for delay_days in scan_delays:
            max_start = min(max_ep_doy - delay_days - window_days + 1, max_ao_doy - window_days + 1)
            for ao_start_doy in range(1, max_start + 1, scan_start_step):
                df = ao_ep_async_table(
                    ao, date_ao, ep_series, date_h,
                    ao_start_doy=ao_start_doy,
                    window_days=window_days,
                    delay_days=delay_days,
                )
                scan_rows.extend(ao_ep_corr_rows(df))
    async_scan = pd.DataFrame(scan_rows)
    async_scan["abs_R"] = async_scan["R_AO_vs_EP100"].abs()
    async_scan["passes_nominal_filter"] = (async_scan["abs_R"] >= 0.5) & (async_scan["P"] < 0.05)
    async_scan = async_scan.sort_values(["passes_nominal_filter", "abs_R", "P"], ascending=[False, False, True])
    async_scan.to_csv(TABLE_DIR / f"{CASE}_AO_lead_EPFlux_lag_window_scan.csv", index=False)

    print("\nTop asynchronous AO -> EPFlux correlations from exploratory scan:")
    print(async_scan.head(15)[[
        "label", "R_AO_vs_EP100", "P", "AO_window", "EP_window", "window_days", "delay_days", "passes_nominal_filter"
    ]])

    def plot_async_heatmaps_for_window(window_days: int):
        scan_window = async_scan.loc[async_scan["window_days"] == window_days].copy()
        fig, axes = plt.subplots(2, 3, figsize=(16.5, 8.8), constrained_layout=True)
        heat_axes = axes.ravel()[:5]
        im = None
        for ax, (metric_col, label, _) in zip(heat_axes, ASYNC_WAVE_COLS):
            metric = metric_col.replace("EP100_", "")
            sub = scan_window.loc[scan_window["metric"] == metric].copy()
            pivot = sub.pivot(index="delay_days", columns="ao_start_doy", values="R_AO_vs_EP100").sort_index()
            im = ax.imshow(
                pivot.values,
                origin="lower",
                aspect="auto",
                cmap="RdBu_r",
                vmin=-1,
                vmax=1,
                extent=[pivot.columns.min(), pivot.columns.max(), pivot.index.min(), pivot.index.max()],
            )
            good = sub.loc[sub["passes_nominal_filter"]]
            if not good.empty:
                ax.scatter(good["ao_start_doy"], good["delay_days"], s=22, facecolors="none", edgecolors="black", linewidths=0.9)
            xticks = [mmdd_to_doy(1, 1), mmdd_to_doy(2, 1), mmdd_to_doy(3, 1), mmdd_to_doy(4, 1)]
            ax.set_xticks([t for t in xticks if pivot.columns.min() <= t <= pivot.columns.max()])
            ax.set_xticklabels([doy_window_label(int(t), 1)[:5] for t in ax.get_xticks()], rotation=30, ha="right")
            ax.set_yticks(scan_delays)
            ax.set_title(f"AO vs {label}")
            ax.set_xlabel("AO window start")
            ax.set_ylabel("EP lag after AO start (days)")
        axes.ravel()[5].axis("off")
        axes.ravel()[5].text(
            0.0, 0.95,
            "Exploratory scan\n"
            f"Shown: {window_days}-day windows\n"
            "Color: Pearson R across 30 members\n"
            "Open circles: |R| >= 0.5 and p < 0.05\n"
            "Only positive delays are tested, so EPFlux is concurrent with or later than AO.",
            va="top", ha="left", fontsize=10,
        )
        fig.colorbar(im, ax=heat_axes, shrink=0.88, label="R(AO, delayed EP100)")
        fig.suptitle(f"AO lead / EPFlux lag scan, {window_days}-day windows", fontsize=12)
        savefig(fig, f"{CASE}_AO_lead_EPFlux_lag_scan_{window_days}day_heatmaps")
        plt.show()

    plot_async_heatmaps_for_window(30)
    plot_async_heatmaps_for_window(20)

    line_window_days = 20
    line_delay_days = 5
    line_df = async_scan.loc[
        (async_scan["window_days"] == line_window_days)
        & (async_scan["delay_days"] == line_delay_days)
    ].copy()
    line_df.to_csv(TABLE_DIR / f"{CASE}_AO_EPFlux_20day_lag5_correlation_lines.csv", index=False)

    fig, ax = plt.subplots(figsize=(10.8, 5.8), constrained_layout=True)
    for metric_col, label, color in ASYNC_WAVE_COLS:
        metric = metric_col.replace("EP100_", "")
        sub = line_df.loc[line_df["metric"] == metric].sort_values("ao_start_doy")
        ax.plot(sub["ao_start_doy"], sub["R_AO_vs_EP100"], marker="o", ms=4.5, lw=1.8, color=color, label=label)
        sig = sub.loc[(sub["R_AO_vs_EP100"].abs() >= 0.5) & (sub["P"] < 0.05)]
        if not sig.empty:
            ax.scatter(sig["ao_start_doy"], sig["R_AO_vs_EP100"], s=64, facecolors="none", edgecolors=color, linewidths=1.4)
    xticks = [mmdd_to_doy(1, 1), mmdd_to_doy(1, 15), mmdd_to_doy(2, 1), mmdd_to_doy(2, 15), mmdd_to_doy(3, 1), mmdd_to_doy(3, 15), mmdd_to_doy(4, 1)]
    ax.set_xticks([t for t in xticks if line_df["ao_start_doy"].min() <= t <= line_df["ao_start_doy"].max()])
    ax.set_xticklabels([doy_window_label(int(t), 1)[:5] for t in ax.get_xticks()], rotation=30, ha="right")
    ax.axhline(0, color="0.25", lw=0.9)
    ax.axhline(0.5, color="0.45", lw=0.8, ls=":")
    ax.axhline(-0.5, color="0.45", lw=0.8, ls=":")
    ax.set_xlabel("AO 20-day window start")
    ax.set_ylabel("R(AO mean, delayed EP100)")
    ax.set_title("20-day windows with EPFlux lagged by 5 days")
    ax.legend(ncol=3, fontsize=9)
    savefig(fig, f"{CASE}_AO_EPFlux_20day_lag5_correlation_lines")
    plt.show()

    jan01_20 = ao_ep_async_table(
        ao, date_ao, ep_series, date_h,
        ao_start_doy=mmdd_to_doy(1, 1),
        window_days=20,
        delay_days=5,
    )
    jan01_20.to_csv(TABLE_DIR / f"{CASE}_AO_Jan01_Jan20_EP_Jan06_Jan25_all_w1plusw2.csv", index=False)
    fig, axes = plt.subplots(1, 2, figsize=(12.6, 5.2), constrained_layout=True)
    scatter_fit(
        axes[0], jan01_20, "AO_mean", "EP100_all_waves",
        "AO Jan01-Jan20 vs all-wave EP100 Jan06-Jan25",
        "AO mean Jan01-Jan20", "EP100 all waves Jan06-Jan25", "black",
        annotate_members=True,
    )
    scatter_fit(
        axes[1], jan01_20, "AO_mean", "EP100_wave1_plus_wave2",
        "AO Jan01-Jan20 vs W1+W2 EP100 Jan06-Jan25",
        "AO mean Jan01-Jan20", "EP100 W1+W2 Jan06-Jan25", "tab:purple",
        annotate_members=True,
    )
    fig.suptitle("AO lead test: 20-day window, 5-day lag; EP100 = mean(-ep2), 100 hPa, 40-80N", fontsize=11)
    savefig(fig, f"{CASE}_AO_Jan01_Jan20_vs_EP_Jan06_Jan25_all_w1plusw2")
    plt.show()

    robust = async_scan.loc[async_scan["passes_nominal_filter"]].copy()
    if robust.empty:
        print("No AO lead / delayed EPFlux combinations passed |R| >= 0.5 and p < 0.05; detailed scatter figure skipped.")
    else:
        top = robust.head(6)
        fig, axes = plt.subplots(2, 3, figsize=(16.0, 9.2), constrained_layout=True)
        color_lookup = {col: color for col, _, color in ASYNC_WAVE_COLS}
        label_lookup = {col: label for col, label, _ in ASYNC_WAVE_COLS}
        for ax, (_, row) in zip(axes.ravel(), top.iterrows()):
            col = f"EP100_{row['metric']}"
            df = ao_ep_async_table(
                ao, date_ao, ep_series, date_h,
                ao_start_doy=int(row["ao_start_doy"]),
                window_days=int(row["window_days"]),
                delay_days=int(row["delay_days"]),
            )
            scatter_fit(
                ax, df, "AO_mean", col,
                f"{label_lookup.get(col, row['label'])}: R={row['R_AO_vs_EP100']:.2f}, p={row['P']:.2g}",
                f"AO {df['AO_window'].iloc[0]}",
                f"EP100 {df['EP_window'].iloc[0]}",
                color_lookup.get(col, "tab:cyan"),
                annotate_members=False,
            )
        fig.suptitle("Top nominally significant asynchronous AO -> EPFlux relationships", fontsize=12)
        savefig(fig, f"{CASE}_AO_lead_EPFlux_top_significant_scatters")
        plt.show()

## 图：AO-EPFlux 任意 window/lag 密集扫描

**做什么**：在 `window >= 20 days`、`lag >= 0 days` 的限制下，对所有整数日 AO window、EPFlux window 和 lag 进行密集扫描，寻找最有潜力的 AO-leading-wave-activity 组合。

**怎么做**：AO 和 EPFlux 均先按成员对齐，然后用逐日累计和快速计算任意窗口平均。每个候选组合定义为：AO 从 `ao_start_doy` 开始平均 `window_days`，EPFlux 从 `ao_start_doy + delay_days` 开始平均同样长度。EPFlux 仍是 `EP100 = mean(-ep2)`，即 `100 hPa, 40-80N` 的垂直 EP-flux 分量，不是 divergence。

**多重检验提醒**：这个扫描会产生大量候选，因此单个最高相关很容易过拟合。输出里同时给 `BH-FDR q` 和局部邻域支持度 `local_support_same_sign`：后者统计同一 metric、同一符号、相近 start/window/delay 里是否也显著。更可信的是一片连续区域，而不是孤立最高点。

**科学问题**：如果 AO 对后续 wave activity 有潜在调制，最佳组合是否集中在物理上可解释的早冬/中冬窗口？是 all-wave、W1+W2、wave2 还是 rest 更稳定？

**运行后解读**：0008-01 的全局局部稳健最高组合集中在 5 月 `wave_rest`，例如 AO May04-May26 vs EP May06-May28，`R≈0.60, p≈4.5e-4`，但 `BH-FDR q≈1`，而且时间太晚，不适合作为早期 source 结论。若限制到 EP 窗口在 2 月底前结束，最有潜力的是早 1 月 AO 与稍晚 `wave2` 的负相关：AO Jan01-Jan21 vs EP Jan05-Jan25，`window=21 d, lag=4 d, R≈-0.789, p≈2.2e-7, q≈0.11`；相邻窗口给出类似结果，W1+W2 也为负、rest 为正。由于任意扫描检验数很大，尚无组合通过严格 `q<0.05`，所以这是探索性候选，不是最终显著结论。

In [ ]:
# -----------------------------
# Dense arbitrary AO-window / EPFlux-lag scan
# -----------------------------
# Search all integer windows with window >= 20 days and non-negative lags.
# To reduce multiple-testing overinterpretation, keep all p<0.05 rows for BH-FDR
# and report a local-support score around the strongest candidates.
from scipy.stats import t as student_t

if ao is None:
    print("Skip dense arbitrary AO/EPFlux scan: projected AO is unavailable.")
else:
    DENSE_MIN_WINDOW_DAYS = 20
    DENSE_KEEP_P_LT = 0.05
    DENSE_KEEP_ABS_R_GE = 0.45
    DENSE_NEIGHBOR_DAYS = 3
    DENSE_MAX_TOP = 300

    def _values_by_member_short(da: xr.DataArray, common_members: Sequence[str]) -> np.ndarray:
        mids = [str(v) for v in da["member_short"].values]
        idx = [mids.index(m) for m in common_members]
        return da.isel(member=idx).values.astype(float)

    common_members = sorted(
        set(str(v) for v in ao["member_short"].values)
        & set(str(v) for v in ep_series["all_waves"]["member_short"].values)
    )
    print(f"Dense scan common members: {len(common_members)}")

    ao_values = _values_by_member_short(ao, common_members)
    ep_values = {wave: _values_by_member_short(ep_series[wave], common_members) for wave in WAVES}
    ep_values["wave1_plus_wave2"] = ep_values["wave1"] + ep_values["wave2"]

    ao_doy = date_to_doy(date_ao[: ao_values.shape[1]])
    ep_doy = date_to_doy(date_h[: ep_values["all_waves"].shape[1]])
    dense_max_doy = int(min(np.nanmax(ao_doy), np.nanmax(ep_doy)))
    dense_min_doy = int(max(np.nanmin(ao_doy), np.nanmin(ep_doy)))
    print(f"Dense scan DOY range: {dense_min_doy}-{dense_max_doy}")

    def _window_cumsums(values: np.ndarray, doys: np.ndarray, max_doy: int):
        sums = np.zeros((values.shape[0], max_doy + 1), dtype=float)
        counts = np.zeros((values.shape[0], max_doy + 1), dtype=float)
        for t_idx, doy in enumerate(np.asarray(doys, dtype=int)):
            if doy < 1 or doy > max_doy:
                continue
            col = values[:, t_idx]
            finite = np.isfinite(col)
            sums[:, doy] += np.where(finite, col, 0.0)
            counts[:, doy] += finite.astype(float)
        return np.cumsum(sums, axis=1), np.cumsum(counts, axis=1)

    def _segment_mean(csum: np.ndarray, ccnt: np.ndarray, start_doy: int, window_days: int) -> np.ndarray:
        end_doy = int(start_doy) + int(window_days) - 1
        total = csum[:, end_doy] - csum[:, start_doy - 1]
        count = ccnt[:, end_doy] - ccnt[:, start_doy - 1]
        out = np.full(total.shape, np.nan, dtype=float)
        np.divide(total, count, out=out, where=count > 0)
        return out

    def _corr_r_p_fast(x: np.ndarray, y: np.ndarray) -> tuple[float, float, int]:
        mask = np.isfinite(x) & np.isfinite(y)
        n = int(mask.sum())
        if n < 3:
            return np.nan, np.nan, n
        xv = x[mask]
        yv = y[mask]
        xv = xv - xv.mean()
        yv = yv - yv.mean()
        denom = math.sqrt(float(np.dot(xv, xv) * np.dot(yv, yv)))
        if denom == 0 or not np.isfinite(denom):
            return np.nan, np.nan, n
        r = float(np.dot(xv, yv) / denom)
        r = max(min(r, 0.999999999), -0.999999999)
        t_stat = abs(r) * math.sqrt((n - 2) / max(1e-15, 1.0 - r * r))
        p = float(2.0 * student_t.sf(t_stat, n - 2))
        return r, p, n

    ao_csum, ao_ccnt = _window_cumsums(ao_values, ao_doy, dense_max_doy)
    ep_cumsums = {key: _window_cumsums(val, ep_doy, dense_max_doy) for key, val in ep_values.items()}

    dense_metric_order = ["all_waves", "wave1", "wave2", "wave_rest", "wave1_plus_wave2"]
    dense_rows = []
    total_tests = 0
    latest_start = dense_max_doy - DENSE_MIN_WINDOW_DAYS + 1

    for window_days in range(DENSE_MIN_WINDOW_DAYS, dense_max_doy - dense_min_doy + 2):
        max_start = dense_max_doy - window_days + 1
        if max_start < dense_min_doy:
            continue
        ao_window_means = {
            start_doy: _segment_mean(ao_csum, ao_ccnt, start_doy, window_days)
            for start_doy in range(dense_min_doy, max_start + 1)
        }
        ep_window_means = {
            metric: {
                start_doy: _segment_mean(ep_cumsums[metric][0], ep_cumsums[metric][1], start_doy, window_days)
                for start_doy in range(dense_min_doy, max_start + 1)
            }
            for metric in dense_metric_order
        }
        for ao_start_doy, ao_vec in ao_window_means.items():
            for ep_start_doy in range(ao_start_doy, max_start + 1):
                delay_days = ep_start_doy - ao_start_doy
                for metric in dense_metric_order:
                    r, p, n = _corr_r_p_fast(ao_vec, ep_window_means[metric][ep_start_doy])
                    total_tests += 1
                    abs_r = abs(r) if np.isfinite(r) else np.nan
                    if np.isfinite(p) and (p < DENSE_KEEP_P_LT or abs_r >= DENSE_KEEP_ABS_R_GE):
                        dense_rows.append({
                            "metric": metric,
                            "label": WAVE_LABELS.get(metric, metric),
                            "R_AO_vs_EP100": r,
                            "P": p,
                            "abs_R": abs_r,
                            "ao_start_doy": ao_start_doy,
                            "ep_start_doy": ep_start_doy,
                            "window_days": window_days,
                            "delay_days": delay_days,
                            "AO_window": doy_window_label(ao_start_doy, window_days),
                            "EP_window": doy_window_label(ep_start_doy, window_days),
                            "n_members": n,
                        })

    dense_filtered = pd.DataFrame(dense_rows)
    print(f"Dense tests evaluated: {total_tests:,}; retained rows: {len(dense_filtered):,}")

    if dense_filtered.empty:
        print("No dense AO/EPFlux candidates passed the retention thresholds.")
    else:
        dense_filtered = dense_filtered.sort_values("P", kind="mergesort").reset_index(drop=True)
        ranks = np.arange(1, len(dense_filtered) + 1, dtype=float)
        q_raw = dense_filtered["P"].values * float(total_tests) / ranks
        q_monotone = np.minimum.accumulate(q_raw[::-1])[::-1]
        dense_filtered["bh_q"] = np.clip(q_monotone, 0, 1)
        dense_filtered["passes_nominal_filter"] = (dense_filtered["abs_R"] >= 0.5) & (dense_filtered["P"] < 0.05)
        dense_filtered["sign"] = np.sign(dense_filtered["R_AO_vs_EP100"]).astype(int)
        dense_filtered["ao_end_doy"] = dense_filtered["ao_start_doy"] + dense_filtered["window_days"] - 1
        dense_filtered["ep_end_doy"] = dense_filtered["ep_start_doy"] + dense_filtered["window_days"] - 1
        dense_filtered["early_source_window"] = dense_filtered["ep_end_doy"] <= mmdd_to_doy(2, 28)

        dense_sig = dense_filtered.loc[dense_filtered["passes_nominal_filter"]].copy()
        top_for_support = dense_sig.sort_values(["abs_R", "P"], ascending=[False, True]).head(DENSE_MAX_TOP).copy()
        support_values = []
        for _, row in top_for_support.iterrows():
            same = dense_sig.loc[
                (dense_sig["metric"] == row["metric"])
                & (dense_sig["sign"] == row["sign"])
                & ((dense_sig["ao_start_doy"] - row["ao_start_doy"]).abs() <= DENSE_NEIGHBOR_DAYS)
                & ((dense_sig["ep_start_doy"] - row["ep_start_doy"]).abs() <= DENSE_NEIGHBOR_DAYS)
                & ((dense_sig["window_days"] - row["window_days"]).abs() <= DENSE_NEIGHBOR_DAYS)
            ]
            support_values.append(int(len(same)))
        top_for_support["local_support_same_sign"] = support_values
        dense_promising = top_for_support.sort_values(
            ["local_support_same_sign", "abs_R", "bh_q", "P"],
            ascending=[False, False, True, True],
        ).reset_index(drop=True)

        dense_early = top_for_support.loc[top_for_support["early_source_window"]].sort_values(
            ["abs_R", "local_support_same_sign", "bh_q", "P"],
            ascending=[False, False, True, True],
        ).reset_index(drop=True)

        filtered_path = TABLE_DIR / f"{CASE}_AO_EPFlux_dense_arbitrary_window_lag_filtered.csv"
        top_path = TABLE_DIR / f"{CASE}_AO_EPFlux_dense_arbitrary_window_lag_top_promising.csv"
        early_path = TABLE_DIR / f"{CASE}_AO_EPFlux_dense_arbitrary_window_lag_early_source_top.csv"
        dense_filtered.to_csv(filtered_path, index=False)
        dense_promising.to_csv(top_path, index=False)
        dense_early.to_csv(early_path, index=False)
        print(f"Saved dense filtered candidates: {filtered_path}")
        print(f"Saved dense top promising candidates: {top_path}")
        print(f"Saved dense early-source candidates: {early_path}")
        print("\nTop dense candidates by local support, |R|, and BH q:")
        print(dense_promising.head(20)[[
            "metric", "R_AO_vs_EP100", "P", "bh_q", "local_support_same_sign",
            "AO_window", "EP_window", "window_days", "delay_days", "n_members"
        ]].to_string(index=False))
        print("\nTop dense early-source candidates by |R|, local support, and BH q (EP window ends by Feb28):")
        print(dense_early.head(20)[[
            "metric", "R_AO_vs_EP100", "P", "bh_q", "local_support_same_sign",
            "AO_window", "EP_window", "window_days", "delay_days", "n_members"
        ]].to_string(index=False))

        def _plot_dense_candidate(best_row: pd.Series, suffix: str, title_prefix: str):
            best_metric = str(best_row["metric"])
            best_window = int(best_row["window_days"])
            best_label = WAVE_LABELS.get(best_metric, best_metric)
            best_col = f"EP100_{best_metric}"
            best_df = ao_ep_async_table(
                ao, date_ao, ep_series, date_h,
                ao_start_doy=int(best_row["ao_start_doy"]),
                window_days=best_window,
                delay_days=int(best_row["delay_days"]),
            )

            max_start = dense_max_doy - best_window + 1
            heat_rows = []
            ao_means_best = {
                start_doy: _segment_mean(ao_csum, ao_ccnt, start_doy, best_window)
                for start_doy in range(dense_min_doy, max_start + 1)
            }
            ep_csum_best, ep_ccnt_best = ep_cumsums[best_metric]
            ep_means_best = {
                start_doy: _segment_mean(ep_csum_best, ep_ccnt_best, start_doy, best_window)
                for start_doy in range(dense_min_doy, max_start + 1)
            }
            for ao_start_doy, ao_vec in ao_means_best.items():
                for ep_start_doy in range(ao_start_doy, max_start + 1):
                    r, p, n = _corr_r_p_fast(ao_vec, ep_means_best[ep_start_doy])
                    heat_rows.append({
                        "ao_start_doy": ao_start_doy,
                        "delay_days": ep_start_doy - ao_start_doy,
                        "R": r,
                        "P": p,
                        "passes": abs(r) >= 0.5 and p < 0.05,
                    })
            heat_df = pd.DataFrame(heat_rows)
            pivot = heat_df.pivot(index="delay_days", columns="ao_start_doy", values="R").sort_index()

            color_lookup = {
                "all_waves": "black", "wave1": "tab:blue", "wave2": "tab:orange",
                "wave_rest": "tab:green", "wave1_plus_wave2": "tab:purple",
            }
            fig, axes = plt.subplots(1, 2, figsize=(14.5, 5.8), constrained_layout=True)
            scatter_fit(
                axes[0], best_df, "AO_mean", best_col,
                f"{title_prefix}: AO vs {best_label}",
                f"AO {best_df['AO_window'].iloc[0]}",
                f"EP100 {best_label} {best_df['EP_window'].iloc[0]}",
                color_lookup.get(best_metric, "tab:blue"),
                annotate_members=True,
            )
            im = axes[1].imshow(
                pivot.values,
                origin="lower",
                aspect="auto",
                cmap="RdBu_r",
                vmin=-1,
                vmax=1,
                extent=[pivot.columns.min(), pivot.columns.max(), pivot.index.min(), pivot.index.max()],
            )
            good = heat_df.loc[heat_df["passes"]]
            if not good.empty:
                axes[1].scatter(good["ao_start_doy"], good["delay_days"], s=18, facecolors="none", edgecolors="black", linewidths=0.8)
            xticks = [mmdd_to_doy(1, 1), mmdd_to_doy(2, 1), mmdd_to_doy(3, 1), mmdd_to_doy(4, 1), mmdd_to_doy(5, 1)]
            axes[1].set_xticks([t for t in xticks if pivot.columns.min() <= t <= pivot.columns.max()])
            axes[1].set_xticklabels([doy_window_label(int(t), 1)[:5] for t in axes[1].get_xticks()], rotation=30, ha="right")
            axes[1].set_xlabel("AO window start")
            axes[1].set_ylabel("EP lag after AO start (days)")
            axes[1].set_title(f"R heatmap: {best_label}, {best_window} d")
            fig.colorbar(im, ax=axes[1], shrink=0.9, label="R(AO, delayed EP100)")
            fig.suptitle(
                f"{title_prefix}; total tests={total_tests:,}\n"
                f"R={best_row['R_AO_vs_EP100']:.2f}, p={best_row['P']:.2g}, q={best_row['bh_q']:.2g}, support={int(best_row['local_support_same_sign'])}",
                fontsize=11,
            )
            savefig(fig, f"{CASE}_AO_EPFlux_dense_arbitrary_window_lag_{suffix}")
            plt.show()

        if not dense_promising.empty:
            _plot_dense_candidate(dense_promising.iloc[0], "best_overall_local_support", "Best overall by local support")
        if not dense_early.empty:
            _plot_dense_candidate(dense_early.iloc[0], "best_early_source", "Best early-source candidate")


## 图：Z300 月平均波型指标与 EPFlux 分量

**做什么**：把原先单一 Jan20-Feb10 的 Z300 source test 扩展到 Jan、Feb、Mar、Apr、May 月平均，分别计算 ACC、stationary-wave closeness、stationary-wave projection 与同月 EPFlux 分量的 member-to-member 相关；同时单独画 monthly stationary-wave projection vs EP100 wave1+wave2 的散点图。

**怎么做**：每个成员的 Z300 为 300 hPa 高度月平均，区域 20-90N。ACC 是成员 Z300 stationary anomaly 与 BWCN0008 同月 Z300 stationary anomaly 的加权 pattern correlation。stationary-wave closeness 是成员 Z300 stationary anomaly 与 B2000WCN001002 全年份同月气候态 stationary wave target 的加权 pattern correlation。projection 是成员异常投影到同一个 B2000WCN 气候态 stationary wave target 上的加权投影系数，即加权 dot(member anomaly, target) / dot(target, target)。若 `Longrun/date_treatment/Climatology.ipynb` 已生成完整的 `Z3_climatology_plev_doy.nc`，这里会优先从 `Z3_clim_all(doy, plev, lat, lon)` 抽取 300 hPa 并按对应月份/窗口平均，再去掉 zonal mean 作为 B2000WCN stationary-wave target；只有该完整气候态缺失时，才回退到旧缓存或从原始 Z3 现算。

**科学问题**：哪个月份的对流层 300 hPa 波型/投影最能对应进入平流层的 EPFlux 分量？这种关系是 wave1、wave2、rest，还是 all-wave 更明显？MAR 若接近 0，是因为算法退化、目标场异常，还是因为该月 projection 与 EP100 W1+W2 的成员排序确实不一致？

**预期**：如果对流层 planetary wave 是源头，Jan-Feb 或 Apr 的 Z300 climatological-stationary-wave projection 应与 100 hPa EPFlux 的 wave1+wave2 更相关；如果 MAR 不是算法问题，MAR 的 projection 和 EP100 W1+W2 应该都有非零 spread，但二者 scatter 没有线性关系。

**运行后解读**：待图生成后填写。Jan20-Feb10 三个 Z300 指标 vs O3 RMSE 的旧图先保留。


In [ ]:
# -----------------------------
# Z300 tropospheric diagnostics
# -----------------------------

def interp_profile_logp(da_var: xr.DataArray, p_hyb: xr.DataArray, p_tgt_pa: float) -> xr.DataArray:
    target = np.array([float(p_tgt_pa)], dtype=float)
    def _interp_col(vcol, pcol):
        vcol = np.asarray(vcol, dtype=float)
        pcol = np.asarray(pcol, dtype=float)
        mask = np.isfinite(vcol) & np.isfinite(pcol) & (pcol > 0)
        if mask.sum() < 2:
            return np.array([np.nan], dtype=float)
        p = pcol[mask]
        v = vcol[mask]
        idx = np.argsort(p)
        return np.interp(np.log(target), np.log(p[idx]), v[idx], left=np.nan, right=np.nan)
    out = xr.apply_ufunc(
        _interp_col,
        da_var,
        p_hyb,
        input_core_dims=[["lev"], ["lev"]],
        output_core_dims=[["plev"]],
        vectorize=True,
        dask="allowed",
        output_dtypes=[float],
    )
    return out.assign_coords(plev=("plev", target)).isel(plev=0, drop=True)


def window_token(label: str) -> str:
    return re.sub(r"[^A-Za-z0-9]+", "_", label).strip("_")


def z300_from_file(path: Path, start_end=Z300_WINDOW, lat_range=LAT_Z300) -> xr.DataArray:
    with xr.open_dataset(path, decode_times=False) as ds:
        date = np.asarray(ds["date"].values, dtype=np.int64)
        mask = date_mask(date, start=start_end[0], end=start_end[1])
        ds = ds.isel(time=mask)
        ds = ds.sel(lat=slice(lat_range[0], lat_range[1]))
        p_mid = ds["hyam"] * ds["P0"] + ds["hybm"] * ds["PS"]
        z = interp_profile_logp(ds["Z3"].transpose("time", "lat", "lon", "lev"), p_mid.transpose("time", "lat", "lon", "lev"), PLEV_Z300_PA)
        z_mean = z.mean("time", skipna=True).load()
    z_mean.name = "Z300"
    z_mean.attrs.update({"units": "m", "plev_pa": PLEV_Z300_PA, "window": str(start_end)})
    return z_mean


def build_z300_hindcast_cache(case=CASE, start_end=Z300_WINDOW, label="Cwindow", overwrite=False):
    out = CACHE_DIR / f"Z300_{case}_members_{window_token(label)}.nc"
    if out.exists() and not overwrite:
        return xr.open_dataarray(out).load()
    files = sorted((HINDCAST_ROOT / case / "Z3").glob("*.Z3.nc"))
    if not files:
        raise FileNotFoundError(f"No Z3 files for {case}")
    das, mids = [], []
    for f in files:
        mid = member_short_id(f.name)
        print("Z300", case, label, mid)
        das.append(z300_from_file(f, start_end=start_end))
        mids.append(mid)
    da = xr.concat(das, dim=pd.Index(mids, name="member"))
    da.to_netcdf(out)
    print(f"Saved cache: {out}")
    return da


def build_z300_bwcn_ref_cache(year=REF_YEAR, start_end=Z300_WINDOW, label="Cwindow", overwrite=False):
    out = CACHE_DIR / f"Z300_BWCN_{year:04d}_{window_token(label)}.nc"
    if out.exists() and not overwrite:
        return xr.open_dataarray(out).load()
    f = BWCN_ROOT / "Z3" / f"BWCN.cam.h3.{year:04d}.Z3.nc"
    da = z300_from_file(f, start_end=start_end)
    da.to_netcdf(out)
    print(f"Saved cache: {out}")
    return da


def doy_values_for_window(start_end):
    month_lengths = np.array([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31], dtype=int)
    (sm, sd), (em, ed) = start_end
    start_doy = int(month_lengths[: sm - 1].sum() + sd)
    end_doy = int(month_lengths[: em - 1].sum() + ed)
    if end_doy < start_doy:
        raise ValueError(f"Window crosses year boundary, unsupported here: {start_end}")
    return np.arange(start_doy, end_doy + 1, dtype=np.int16)


def z300_target_from_full_z3_climatology(clim_file: Path, label, start_end) -> xr.DataArray:
    doys = doy_values_for_window(start_end)
    stat = clim_file.stat()
    with xr.open_dataset(clim_file, decode_times=False) as ds:
        if "Z3_clim_all" not in ds.data_vars:
            raise KeyError(f"{clim_file} does not contain Z3_clim_all")
        z = ds["Z3_clim_all"].sel(plev=PLEV_Z300_PA, method="nearest")
        available = np.intersect1d(np.asarray(z["doy"].values, dtype=np.int16), doys)
        if len(available) == 0:
            raise ValueError(f"No requested DOYs {doys[0]}-{doys[-1]} found in {clim_file}")
        z = z.sel(doy=available).mean("doy", skipna=True)
        z = select_latband(z, LAT_Z300).load()
    target = (z - z.mean("lon", skipna=True)).astype(np.float32)
    target.name = f"Z300_B2000_stationary_wave_target_{window_token(label)}"
    target.attrs.update({
        "definition": "B2000WCN001002 all-year Z3_clim_all at 300 hPa averaged over the requested window; zonal mean removed",
        "source_file": str(clim_file),
        "source_file_size": int(stat.st_size),
        "source_file_mtime_ns": int(stat.st_mtime_ns),
        "source_variable": "Z3_clim_all",
        "window_label": str(label),
        "start_end": str(start_end),
        "doy_start": int(doys[0]),
        "doy_end": int(doys[-1]),
        "plev_pa": float(PLEV_Z300_PA),
        "lat_range": str(LAT_Z300),
    })
    return target


def cache_matches_full_z3_climatology(cache_file: Path, source_file: Path, start_end) -> bool:
    if not cache_file.exists() or cache_file.stat().st_size <= 0:
        return False
    try:
        stat = source_file.stat()
        with xr.open_dataarray(cache_file, decode_times=False) as da:
            return (
                da.attrs.get("source_file") == str(source_file)
                and da.attrs.get("start_end") == str(start_end)
                and int(da.attrs.get("source_file_size", -1)) == int(stat.st_size)
                and int(da.attrs.get("source_file_mtime_ns", -1)) == int(stat.st_mtime_ns)
            )
    except Exception:
        return False


def build_z300_b2000_stationary_target(label, start_end, overwrite=False, max_years=CLIM_MAX_B2000_YEARS_FOR_Z300):
    out = CACHE_DIR / f"Z300_B2000WCN001002_{window_token(label)}_stationary_wave_target.nc"
    full_z3_clim_file = B2000_ROOT / "climatology" / "Z3_climatology_plev_doy.nc"

    if full_z3_clim_file.exists():
        if not overwrite and cache_matches_full_z3_climatology(out, full_z3_clim_file, start_end):
            return xr.open_dataarray(out).load()
        target = z300_target_from_full_z3_climatology(full_z3_clim_file, label, start_end)
        target.to_netcdf(out)
        print(f"Saved cache from full Z3 climatology: {out}")
        return target

    if out.exists() and not overwrite:
        return xr.open_dataarray(out).load()

    monthly_clim_file = B2000_ROOT / "climatology" / "Z300_monthly_stationary_wave_climatology.nc"
    if label in MONTH_ORDER and monthly_clim_file.exists():
        month_num = MONTH_ORDER.index(label) + 1
        with xr.open_dataset(monthly_clim_file, decode_times=False) as ds:
            target = ds["Z300_stationary_wave"].sel(month=month_num).load()
        target.name = f"Z300_B2000_stationary_wave_target_{window_token(label)}"
        target.attrs.update({
            "definition": "Legacy B2000WCN001002 monthly 300 hPa stationary-wave target; zonal mean removed",
            "source_file": str(monthly_clim_file),
            "month": label,
        })
        target.to_netcdf(out)
        print(f"Saved cache from legacy monthly Z300 climatology: {out}")
        return target

    files = sorted((B2000_ROOT / "Z3").glob("B2000WCN.sample.cam.h3.*.Z3.nc"))
    if max_years is not None:
        files = files[: int(max_years)]
    if not files:
        raise FileNotFoundError(f"No B2000WCN001002 Z3 files under {B2000_ROOT / 'Z3'}")
    das = []
    for f in files:
        print("Z300 B2000 climatology", label, f.name)
        das.append(z300_from_file(f, start_end=start_end, lat_range=LAT_Z300))
    clim = xr.concat(das, dim="year_index").mean("year_index", skipna=True)
    target = clim - clim.mean("lon", skipna=True)
    target.name = f"Z300_B2000_stationary_wave_target_{window_token(label)}"
    target.attrs.update({
        "definition": "Fallback B2000WCN001002 all-year climatological stationary wave at 300 hPa; zonal mean removed",
        "window": str(start_end),
        "lat_range": str(LAT_Z300),
        "source_root": str(B2000_ROOT),
        "max_years": "all" if max_years is None else int(max_years),
    })
    target.to_netcdf(out)
    print(f"Saved cache from raw Z3 fallback: {out}")
    return target

def weighted_pattern_corr(a: xr.DataArray, b: xr.DataArray, lat_range=LAT_Z300, remove_zonal_mean=True):
    a, b = xr.align(select_latband(a, lat_range), select_latband(b, lat_range), join="inner")
    if remove_zonal_mean:
        a = a - a.mean("lon", skipna=True)
        b = b - b.mean("lon", skipna=True)
    w = np.sqrt(np.cos(np.deg2rad(a["lat"])).clip(0, 1))
    aw = (a * w).values.ravel()
    bw = (b * w).values.ravel()
    mask = np.isfinite(aw) & np.isfinite(bw)
    if mask.sum() < 10:
        return np.nan
    return float(np.corrcoef(aw[mask], bw[mask])[0, 1])


def weighted_projection(a: xr.DataArray, target: xr.DataArray, lat_range=LAT_Z300):
    a, target = xr.align(select_latband(a, lat_range), select_latband(target, lat_range), join="inner")
    a = a - a.mean("lon", skipna=True)
    target = target - target.mean("lon", skipna=True)
    w = np.cos(np.deg2rad(a["lat"])).clip(0, 1)
    num = (a * target * w).sum(skipna=True)
    den = (target * target * w).sum(skipna=True)
    return float((num / den).values)


def ep_wide_for_window(start_end):
    df, _ = load_all_wave_metrics(CASE, date_h, start_end=start_end)
    wide = df.pivot(index="member", columns="wave", values="ep100_upward").reset_index()
    wide = wide.rename(columns={w: f"EP100_{w}" for w in WAVES})
    wide["EP100_wave1_plus_wave2"] = wide["EP100_wave1"] + wide["EP100_wave2"]
    return wide


def z300_metric_table_for_window(label, start_end):
    members = build_z300_hindcast_cache(CASE, start_end=start_end, label=label)
    ref = build_z300_bwcn_ref_cache(REF_YEAR, start_end=start_end, label=label)
    target = build_z300_b2000_stationary_target(label, start_end)
    rows = []
    for mid in members["member"].values:
        z = members.sel(member=mid)
        rows.append({
            "member": str(mid),
            "window": label,
            "Z300_ACC_to_BWCN0008": weighted_pattern_corr(z, ref),
            "Z300_stationary_closeness": weighted_pattern_corr(z, target),
            "Z300_stationary_projection": weighted_projection(z, target),
        })
    return pd.DataFrame(rows)


def z300_ep_correlation_rows(metric_df, ep_df, window_label):
    joined = metric_df.merge(ep_df, on="member", how="inner")
    rows = []
    for z_col in ["Z300_ACC_to_BWCN0008", "Z300_stationary_closeness", "Z300_stationary_projection"]:
        for ep_col, key, ep_label, _ in EP_SCALAR_COLS:
            r, p = finite_corr(joined[z_col], joined[ep_col])
            rows.append({"window": window_label, "z300_metric": z_col, "ep_metric": key, "R": r, "P": p})
    return rows, joined


def plot_monthly_z300_heatmap(corr_df, z_metric, title, filename):
    sub = corr_df[corr_df["z300_metric"] == z_metric].copy()
    mat = sub.pivot(index="window", columns="ep_metric", values="R").reindex(index=MONTH_ORDER)
    mat = mat[["all_waves", "wave1", "wave2", "wave_rest", "wave1_plus_wave2"]]
    fig, ax = plt.subplots(figsize=(8.8, 4.8), constrained_layout=True)
    im = ax.imshow(mat.values, cmap="RdBu_r", vmin=-0.8, vmax=0.8, aspect="auto")
    ax.set_xticks(np.arange(mat.shape[1]))
    ax.set_xticklabels(["All", "W1", "W2", "Rest", "W1+W2"])
    ax.set_yticks(np.arange(mat.shape[0]))
    ax.set_yticklabels(mat.index.tolist())
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            val = mat.values[i, j]
            ax.text(j, i, f"{val:.2f}" if np.isfinite(val) else "NA", ha="center", va="center", fontsize=9)
    ax.set_title(title)
    ax.set_xlabel("EP100 component in same month")
    ax.set_ylabel("Z300 monthly mean")
    fig.colorbar(im, ax=ax, label="Across-member correlation R")
    savefig(fig, filename)
    plt.show()


def z300_monthly_projection_sanity(monthly_values: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for month in MONTH_ORDER:
        sub = monthly_values[monthly_values["window"] == month].copy()
        x = sub["Z300_stationary_projection"].astype(float)
        y = sub["EP100_wave1_plus_wave2"].astype(float)
        r, p = finite_corr(x, y)
        target = build_z300_b2000_stationary_target(month, MONTH_WINDOWS[month])
        target_eddy = target - target.mean("lon", skipna=True)
        rows.append({
            "window": month,
            "n_members": int(np.isfinite(x).sum()),
            "projection_mean": float(np.nanmean(x)),
            "projection_std": float(np.nanstd(x, ddof=1)),
            "projection_min": float(np.nanmin(x)),
            "projection_max": float(np.nanmax(x)),
            "ep100_wave1_plus_wave2_mean": float(np.nanmean(y)),
            "ep100_wave1_plus_wave2_std": float(np.nanstd(y, ddof=1)),
            "target_stationary_std_m": float(np.nanstd(target_eddy.values)),
            "R_projection_vs_EP100_wave1_plus_wave2": r,
            "P_projection_vs_EP100_wave1_plus_wave2": p,
            "sanity_note": "Near-zero R is acceptable if projection_std, EP_std, and target_std are non-zero.",
        })
    return pd.DataFrame(rows)


def plot_stationary_projection_wave12_scatter(monthly_values: pd.DataFrame, months=("Jan", "Feb", "Mar", "Apr")):
    ncols = 2
    nrows = int(np.ceil(len(months) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(10.8, 8.2), constrained_layout=True)
    axes = np.asarray(axes).ravel()
    for ax, month in zip(axes, months):
        sub = monthly_values[monthly_values["window"] == month].copy()
        scatter_fit(
            ax,
            sub,
            "Z300_stationary_projection",
            "EP100_wave1_plus_wave2",
            title=month,
            xlabel="Z300 projection",
            ylabel="EP100 W1+W2",
            color="tab:purple",
            annotate_members=False,
        )
    for ax in axes[len(months):]:
        ax.set_visible(False)
    fig.suptitle("Monthly stationary-wave projection vs same-month EP100 W1+W2", fontsize=11)
    savefig(fig, f"{CASE}_Z300_monthly_stationary_projection_vs_EP100_wave1plus2_scatter")
    plt.show()


if RUN_Z300_DIAGNOSTICS:
    monthly_corr_rows = []
    monthly_joined = []
    for month in MONTH_ORDER:
        metric_df = z300_metric_table_for_window(month, MONTH_WINDOWS[month])
        ep_df = ep_wide_for_window(MONTH_WINDOWS[month])
        rows, joined = z300_ep_correlation_rows(metric_df, ep_df, month)
        monthly_corr_rows.extend(rows)
        monthly_joined.append(joined.assign(window=month))
    z300_monthly_corr = pd.DataFrame(monthly_corr_rows)
    z300_monthly_values = pd.concat(monthly_joined, ignore_index=True)
    z300_monthly_corr.to_csv(TABLE_DIR / f"{CASE}_Z300_monthly_metric_EPFlux_correlations.csv", index=False)
    z300_monthly_values.to_csv(TABLE_DIR / f"{CASE}_Z300_monthly_metric_EPFlux_values.csv", index=False)
    print(z300_monthly_corr.head())

    z300_projection_sanity = z300_monthly_projection_sanity(z300_monthly_values)
    z300_projection_sanity.to_csv(TABLE_DIR / f"{CASE}_Z300_monthly_stationary_projection_sanity.csv", index=False)
    print("\nStationary projection sanity check:")
    print(z300_projection_sanity[[
        "window",
        "projection_std",
        "ep100_wave1_plus_wave2_std",
        "target_stationary_std_m",
        "R_projection_vs_EP100_wave1_plus_wave2",
    ]])

    plot_monthly_z300_heatmap(
        z300_monthly_corr,
        "Z300_ACC_to_BWCN0008",
        "Monthly Z300 ACC vs EP100 components",
        f"{CASE}_Z300_ACC_vs_EPFlux_components",
    )
    plot_monthly_z300_heatmap(
        z300_monthly_corr,
        "Z300_stationary_closeness",
        "Monthly stationary-wave closeness vs EP100",
        f"{CASE}_Z300_stationary_closeness_vs_EPFlux_components",
    )
    plot_monthly_z300_heatmap(
        z300_monthly_corr,
        "Z300_stationary_projection",
        "Monthly stationary-wave projection vs EP100",
        f"{CASE}_Z300_stationary_projection_vs_EPFlux_components",
    )
    plot_stationary_projection_wave12_scatter(z300_monthly_values, months=("Jan", "Feb", "Mar", "Apr"))

    # Keep the original Jan20-Feb10 source metrics vs O3 RMSE diagnostic for now.
    c_metric = z300_metric_table_for_window("Jan20_Feb10", Z300_WINDOW)
    z300_join = rmse_ep_all.merge(c_metric.drop(columns=["window"]), on="member", how="left")
    z300_join["EP100_wave1_plus_wave2"] = z300_join["EP100_wave1"] + z300_join["EP100_wave2"]
    z300_join.to_csv(TABLE_DIR / f"{CASE}_Z300_tropospheric_metrics.csv", index=False)
    print(z300_join.head())

    fig, axes = plt.subplots(1, 3, figsize=(13.8, 4.8), constrained_layout=True)
    rmse_specs = [
        ("Z300_ACC_to_BWCN0008", "ACC", "tab:olive"),
        ("Z300_stationary_closeness", "Closeness", "tab:cyan"),
        ("Z300_stationary_projection", "Projection", "tab:brown"),
    ]
    for ax, (xcol, label, color) in zip(axes, rmse_specs):
        scatter_fit(ax, z300_join, xcol, "RMSE_DU", title=f"{label} vs RMSE", xlabel=label, ylabel="O3 RMSE (DU)", color=color, annotate_members=False)
    fig.suptitle("Jan20-Feb10 Z300 source metrics vs O3 RMSE", fontsize=11)
    savefig(fig, f"{CASE}_Z300_tropospheric_signal_tests")
    plt.show()
else:
    print("RUN_Z300_DIAGNOSTICS is False; skipping Z300 cache build and scatter plots.")


## 图：Z300 stationary projection 30天窗口与滞后 EPFlux

**做什么**：把 `Z300_stationary_projection_vs_EPFlux_components` 从月平均拓展为 30 天滑动窗口，并测试 EPFlux 相对 Z300 stationary projection 滞后 `0/5/10/15/20/30/40 days` 时的相关性；同时单独输出 Jan 和 Feb 的 stationary projection vs W1+W2 散点图。

**怎么做**：每个 30 天窗口先计算成员 Z300 stationary projection：成员 300 hPa Z3 在 `20-90N` 去 zonal mean 后，投影到同一 30 天窗口的 B2000WCN001002 climatological stationary-wave target 上。EPFlux 指标保持一致：`EP100 = mean(-ep2)`，`100 hPa`、`40-80N`、同样 30 天窗口平均，不是 divergence。lag 定义为 `EP window start = Z300 window start + lag_days`。

**科学问题**：如果对流层 300 hPa stationary-wave-like pattern 是后续进入平流层 wave activity 的源头，那么 projection 应该领先或同期对应更强的 EP100 W1+W2，且这种关系应在相邻 start/lag 上形成连续区域。

**预期**：优先看 W1+W2，其次看 wave1/wave2 分量是否解释 all-wave；Jan/FEB 月平均散点用于和前面的 monthly 图保持可解释链接。

**运行后解读**：验证运行中，最强相关主要出现在 Apr 起始窗口：All-wave 同期最高约 `R=0.68`，W1+W2 同期最高约 `R=0.65`；Jan/FEB 月散点中 Feb W1+W2 相关约 `R=0.41, p=0.025`，Jan 约 `R=0.33, p=0.071`。因此这张图目前更像是在提示 late-winter/spring stationary projection 与 EP100 的关系更稳定，Jan/FEB 需要结合具体散点和物理解释谨慎看。

In [ ]:
# -----------------------------
# Z300 stationary projection lead / delayed EPFlux tests
# -----------------------------
# Reuse the same stationary-projection definition as the monthly Z300 block:
# member Z300 stationary anomaly projected onto the B2000WCN001002 stationary-wave target.
# This implementation caches daily Z300 once and then slices 30-day windows from that cache.

Z300_PROJ_WINDOW_DAYS = 30
Z300_PROJ_SCAN_DELAYS = [0, 5, 10, 15, 20, 30, 40]
Z300_PROJ_SCAN_START_STEP = 5
Z300_PROJ_DAILY_RANGE = ((1, 1), (5, 30))
Z300_PROJ_LEV_SLICE = slice(52, 61)  # Hybrid levels bracketing 300 hPa for typical polar PS values.

# Keep this block runnable even when the earlier AO-lag block was skipped.
if "_DOY_MONTH_START" not in globals():
    _DOY_MONTH_START = np.array([0, 31, 59, 90, 120, 151, 181, 212, 243, 273, 304, 334], dtype=int)
    _MONTH_ABBR = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

if "mmdd_to_doy" not in globals():
    def mmdd_to_doy(month: int, day: int) -> int:
        return int(_DOY_MONTH_START[int(month) - 1] + int(day))

if "date_to_doy" not in globals():
    def date_to_doy(date_values) -> np.ndarray:
        _, month, day = date_parts(date_values)
        month = np.asarray(month, dtype=int)
        day = np.asarray(day, dtype=int)
        return _DOY_MONTH_START[month - 1] + day

if "doy_to_mmdd" not in globals():
    def doy_to_mmdd(doy: int) -> Tuple[int, int]:
        doy = int(doy)
        month = int(np.searchsorted(_DOY_MONTH_START + 1, doy, side="right"))
        month = min(max(month, 1), 12)
        day = doy - int(_DOY_MONTH_START[month - 1])
        return month, day

if "doy_window_label" not in globals():
    def doy_window_label(start_doy: int, window_days: int) -> str:
        sm, sd = doy_to_mmdd(start_doy)
        em, ed = doy_to_mmdd(start_doy + window_days - 1)
        return f"{_MONTH_ABBR[sm - 1]}{sd:02d}-{_MONTH_ABBR[em - 1]}{ed:02d}"

if "EP_SCALAR_COLS" not in globals():
    EP_SCALAR_COLS = [
        ("EP100_all_waves", "all_waves", "All", "black"),
        ("EP100_wave1", "wave1", "W1", "tab:blue"),
        ("EP100_wave2", "wave2", "W2", "tab:orange"),
        ("EP100_wave_rest", "wave_rest", "Rest", "tab:green"),
        ("EP100_wave1_plus_wave2", "wave1_plus_wave2", "W1+W2", "tab:purple"),
    ]

if "date_h" not in globals():
    o3_h, date_h = load_hindcast_o3(CASE)


def z300_doy_to_start_end(start_doy: int, window_days: int) -> Tuple[Tuple[int, int], Tuple[int, int]]:
    sm, sd = doy_to_mmdd(start_doy)
    em, ed = doy_to_mmdd(start_doy + window_days - 1)
    return (sm, sd), (em, ed)


def build_z300_hindcast_daily_cache(case=CASE, start_end=Z300_PROJ_DAILY_RANGE, label="JanMay", overwrite=False) -> xr.DataArray:
    out = CACHE_DIR / f"Z300_{case}_members_daily_{window_token(label)}.nc"
    if out.exists() and not overwrite:
        return xr.open_dataarray(out).load()
    files = sorted((HINDCAST_ROOT / case / "Z3").glob("*.Z3.nc"))
    if not files:
        raise FileNotFoundError(f"No Z3 files for {case}")
    das, mids = [], []
    for f in files:
        mid = member_short_id(f.name)
        print("Z300 daily", case, label, mid, flush=True)
        with xr.open_dataset(f, decode_times=False) as ds:
            date = np.asarray(ds["date"].values, dtype=np.int64)
            mask = date_mask(date, start=start_end[0], end=start_end[1])
            sub = ds.isel(time=mask, lev=Z300_PROJ_LEV_SLICE).sel(lat=slice(LAT_Z300[0], LAT_Z300[1]))
            p_mid = sub["hyam"] * sub["P0"] + sub["hybm"] * sub["PS"]
            z = interp_profile_logp(
                sub["Z3"].transpose("time", "lat", "lon", "lev"),
                p_mid.transpose("time", "lat", "lon", "lev"),
                PLEV_Z300_PA,
            )
            z = z.assign_coords(date=("time", date[mask])).load()
        das.append(z.astype(np.float32))
        mids.append(mid)
    da = xr.concat(das, dim=pd.Index(mids, name="member"))
    if "time" in da.dims:
        da = da.rename({"time": "lead_time"})
    da.name = "Z300"
    da.attrs.update({
        "definition": "Daily 300 hPa geopotential height interpolated from Hindcast Z3; used for moving-window stationary projection tests",
        "units": "m",
        "plev_pa": float(PLEV_Z300_PA),
        "lat_range": str(LAT_Z300),
        "date_range": str(start_end),
        "hybrid_lev_slice": "52:61",
    })
    da.to_netcdf(out)
    print(f"Saved cache: {out}")
    return da


def z300_window_mean_from_daily(daily: xr.DataArray, start_doy: int, window_days: int) -> xr.DataArray:
    doys = np.asarray(date_to_doy(daily["date"].values), dtype=int)
    mask = (doys >= int(start_doy)) & (doys <= int(start_doy + window_days - 1))
    if not np.any(mask):
        raise ValueError(f"No Z300 dates found for {doy_window_label(start_doy, window_days)}")
    return daily.isel(lead_time=mask).mean("lead_time", skipna=True).load()


def z300_projection_for_doy_window(start_doy: int, window_days: int = Z300_PROJ_WINDOW_DAYS) -> pd.DataFrame:
    label = f"D{start_doy:03d}_{window_days}d"
    members = z300_window_mean_from_daily(_Z300_PROJ_DAILY, start_doy, window_days)
    target = build_z300_b2000_stationary_target(label, z300_doy_to_start_end(start_doy, window_days))
    rows = []
    for mid in members["member"].values:
        rows.append({
            "member": str(mid),
            "window": label,
            "Z300_stationary_projection": weighted_projection(members.sel(member=mid), target),
        })
    return pd.DataFrame(rows).assign(
        z_start_doy=int(start_doy),
        z_end_doy=int(start_doy + window_days - 1),
        z_window_label=doy_window_label(start_doy, window_days),
    )


def build_ep100_wave_cache(case=CASE):
    cache = {}
    for wave in WAVES:
        da = load_epflux_wave(case, wave=wave)
        da = da.isel(lead_time=slice(0, len(date_h)))
        da = da.assign_coords(date=("lead_time", np.asarray(date_h[: da.sizes["lead_time"]], dtype=np.int64)))
        cache[wave] = da
    return cache


def ep_wide_for_doy_window(start_doy: int, window_days: int = Z300_PROJ_WINDOW_DAYS) -> pd.DataFrame:
    rows = []
    for wave, da in _Z300_PROJ_EP_CACHE.items():
        doys = np.asarray(date_to_doy(da["date"].values), dtype=int)
        mask = (doys >= int(start_doy)) & (doys <= int(start_doy + window_days - 1))
        metric = da.isel(lead_time=mask).mean("lead_time", skipna=True)
        for member, val in zip(metric["member_short"].values, metric.values):
            rows.append({"member": str(member), "wave": wave, "ep100_upward": float(val)})
    out = pd.DataFrame(rows).pivot(index="member", columns="wave", values="ep100_upward").reset_index()
    out = out.rename(columns={w: f"EP100_{w}" for w in WAVES})
    out["EP100_wave1_plus_wave2"] = out["EP100_wave1"] + out["EP100_wave2"]
    return out.assign(
        ep_start_doy=int(start_doy),
        ep_end_doy=int(start_doy + window_days - 1),
        ep_window_label=doy_window_label(start_doy, window_days),
    )


_Z300_PROJ_DAILY = build_z300_hindcast_daily_cache(CASE, Z300_PROJ_DAILY_RANGE, label="JanMay")
_Z300_PROJ_EP_CACHE = build_ep100_wave_cache(CASE)

max_ep_doy = int(np.nanmax(date_to_doy(date_h)))
max_start = max_ep_doy - Z300_PROJ_WINDOW_DAYS + 1
z300_proj_rows = []
z300_proj_joined = []
_projection_cache: dict[int, pd.DataFrame] = {}
_ep_cache: dict[int, pd.DataFrame] = {}

for z_start_doy in range(1, max_start + 1, Z300_PROJ_SCAN_START_STEP):
    if z_start_doy not in _projection_cache:
        _projection_cache[z_start_doy] = z300_projection_for_doy_window(z_start_doy, Z300_PROJ_WINDOW_DAYS)
    proj_df = _projection_cache[z_start_doy]
    for delay_days in Z300_PROJ_SCAN_DELAYS:
        ep_start_doy = z_start_doy + delay_days
        if ep_start_doy > max_start:
            continue
        if ep_start_doy not in _ep_cache:
            _ep_cache[ep_start_doy] = ep_wide_for_doy_window(ep_start_doy, Z300_PROJ_WINDOW_DAYS)
        ep_df = _ep_cache[ep_start_doy]
        joined = proj_df.merge(ep_df, on="member", how="inner")
        joined = joined.assign(delay_days=int(delay_days))
        z300_proj_joined.append(joined)
        for ep_col, key, ep_label, _ in EP_SCALAR_COLS:
            r, p = finite_corr(joined["Z300_stationary_projection"], joined[ep_col])
            z300_proj_rows.append({
                "z_start_doy": int(z_start_doy),
                "ep_start_doy": int(ep_start_doy),
                "window_days": int(Z300_PROJ_WINDOW_DAYS),
                "delay_days": int(delay_days),
                "z_window": doy_window_label(z_start_doy, Z300_PROJ_WINDOW_DAYS),
                "ep_window": doy_window_label(ep_start_doy, Z300_PROJ_WINDOW_DAYS),
                "ep_metric": key,
                "ep_label": ep_label,
                "R": r,
                "P": p,
                "abs_R": abs(r) if np.isfinite(r) else np.nan,
                "passes_nominal_filter": bool(np.isfinite(r) and abs(r) >= 0.5 and p < 0.05),
                "n_members": int(joined[["Z300_stationary_projection", ep_col]].dropna().shape[0]),
            })

z300_projection_lag_corr = pd.DataFrame(z300_proj_rows).sort_values(
    ["passes_nominal_filter", "abs_R", "P"], ascending=[False, False, True]
)
z300_projection_lag_values = pd.concat(z300_proj_joined, ignore_index=True) if z300_proj_joined else pd.DataFrame()
z300_projection_lag_corr.to_csv(TABLE_DIR / f"{CASE}_Z300_stationary_projection_30day_lag_EPFlux_correlations.csv", index=False)
z300_projection_lag_values.to_csv(TABLE_DIR / f"{CASE}_Z300_stationary_projection_30day_lag_EPFlux_values.csv", index=False)
print("Top Z300 stationary projection lead / EPFlux lag correlations:")
print(z300_projection_lag_corr.head(15)[[
    "ep_label", "R", "P", "z_window", "ep_window", "delay_days", "passes_nominal_filter"
]].to_string(index=False))

fig, axes = plt.subplots(2, 3, figsize=(16.5, 8.8), constrained_layout=True)
heat_axes = axes.ravel()[:5]
im = None
for ax, (ep_col, key, ep_label, _) in zip(heat_axes, EP_SCALAR_COLS):
    sub = z300_projection_lag_corr.loc[z300_projection_lag_corr["ep_metric"] == key].copy()
    pivot = sub.pivot(index="delay_days", columns="z_start_doy", values="R").sort_index()
    im = ax.imshow(
        pivot.values,
        origin="lower",
        aspect="auto",
        cmap="RdBu_r",
        vmin=-1,
        vmax=1,
        extent=[pivot.columns.min(), pivot.columns.max(), pivot.index.min(), pivot.index.max()],
    )
    good = sub.loc[sub["passes_nominal_filter"]]
    if not good.empty:
        ax.scatter(good["z_start_doy"], good["delay_days"], s=22, facecolors="none", edgecolors="black", linewidths=0.9)
    xticks = [mmdd_to_doy(1, 1), mmdd_to_doy(2, 1), mmdd_to_doy(3, 1), mmdd_to_doy(4, 1)]
    ax.set_xticks([t for t in xticks if pivot.columns.min() <= t <= pivot.columns.max()])
    ax.set_xticklabels([doy_window_label(int(t), 1)[:5] for t in ax.get_xticks()], rotation=30, ha="right")
    ax.set_yticks(Z300_PROJ_SCAN_DELAYS)
    ax.set_title(f"Projection vs {ep_label}")
    ax.set_xlabel("Z300 projection window start")
    ax.set_ylabel("EP lag after Z300 start (days)")
axes.ravel()[5].axis("off")
axes.ravel()[5].text(
    0.0, 0.95,
    "Z300 stationary projection lag scan\n"
    "Shown: 30-day windows\n"
    "Projection: member Z300 stationary anomaly onto B2000WCN target\n"
    "EP100: mean(-ep2), 100 hPa, 40-80N\n"
    "Open circles: |R| >= 0.5 and p < 0.05",
    va="top", ha="left", fontsize=9.5,
)
fig.colorbar(im, ax=heat_axes, shrink=0.88, label="R(Z300 projection, delayed EP100)")
fig.suptitle("Z300 stationary projection lead / EPFlux lag scan, 30-day windows", fontsize=12)
savefig(fig, f"{CASE}_Z300_stationary_projection_lead_EPFlux_lag_scan_30day_heatmaps")
plt.show()

# Direct month-level Jan/FEB scatter pair requested for projection vs W1+W2.
def z300_projection_for_named_window(label: str, start_end) -> pd.DataFrame:
    start_doy = mmdd_to_doy(*start_end[0])
    end_doy = mmdd_to_doy(*start_end[1])
    window_days = end_doy - start_doy + 1
    out = z300_projection_for_doy_window(start_doy, window_days)
    return out.drop(columns=["window"]).assign(window=label)


def ep_wide_for_named_window(label: str, start_end) -> pd.DataFrame:
    start_doy = mmdd_to_doy(*start_end[0])
    end_doy = mmdd_to_doy(*start_end[1])
    window_days = end_doy - start_doy + 1
    return ep_wide_for_doy_window(start_doy, window_days).assign(window=label)

jan_feb_frames = []
for month in ["Jan", "Feb"]:
    metric_df = z300_projection_for_named_window(month, MONTH_WINDOWS[month])
    ep_df = ep_wide_for_named_window(month, MONTH_WINDOWS[month])
    jan_feb_frames.append(metric_df.merge(ep_df, on=["member", "window"], how="inner"))
jan_feb_projection = pd.concat(jan_feb_frames, ignore_index=True)
jan_feb_projection.to_csv(TABLE_DIR / f"{CASE}_Z300_Jan_Feb_stationary_projection_vs_EP100_wave1plus2.csv", index=False)
fig, axes = plt.subplots(1, 2, figsize=(12.2, 5.2), constrained_layout=True)
for ax, month in zip(axes, ["Jan", "Feb"]):
    sub = jan_feb_projection.loc[jan_feb_projection["window"] == month].copy()
    scatter_fit(
        ax,
        sub,
        "Z300_stationary_projection",
        "EP100_wave1_plus_wave2",
        title=f"{month}: Z300 projection vs EP100 W1+W2",
        xlabel=f"{month} Z300 stationary projection",
        ylabel=f"{month} EP100 W1+W2",
        color="tab:purple",
        annotate_members=True,
    )
fig.suptitle("Monthly Z300 stationary projection vs same-month EP100 W1+W2", fontsize=11)
savefig(fig, f"{CASE}_Z300_Jan_Feb_stationary_projection_vs_EP100_wave1plus2_scatter")
plt.show()


## 图：Hindcast O3/U 演变与 JAN INITIAL ONLY best/worst

**做什么**：复刻 poster-style 的 Hindcast O3 column evolution，并加入 60N 10 hPa zonal wind；JAN INITIAL ONLY 图中同时画 all-30 mean、best5 mean、worst5 mean。

**怎么做**：best/worst 按 Jan1-May30 的 O3 RMSE 排序。O3 使用 cleaned partial O3；U60N10 从 raw U 插值到 10 hPa 并做 zonal mean。

**科学问题**：可预报性好的/差的成员在臭氧和极夜急流演变上何时分离？

**预期**：worst5 可能更早出现波驱动相关的风场/臭氧偏离。

**运行后解读**：待图生成后填写。


In [ ]:
# -----------------------------
# Poster-style evolution plots from cleaned products.
# O3 uses new partial_O3. U60N10 uses raw Hindcast U and a local cache.
# This block also draws the JAN INITIAL ONLY all-30, best-5, and worst-5 means on the same plot.
# Best/worst are ranked by full-window O3 RMSE from the cleaned partial_O3 product.
# -----------------------------

ranked_members = rmse_ep_all.sort_values("RMSE_DU")["member"].tolist()
best5_members = ranked_members[:5]
worst5_members = ranked_members[-5:]
print("JAN INITIAL ONLY best5 by O3 RMSE:", best5_members)
print("JAN INITIAL ONLY worst5 by O3 RMSE:", worst5_members)


def subset_members_da(da: xr.DataArray, members: Sequence[str]) -> xr.DataArray:
    if "member" not in da.dims:
        return da
    member_vals = [str(v) for v in da["member"].values]
    keep_idx = []
    requested = set(members)
    for i, v in enumerate(member_vals):
        short = member_short_id(v)
        if short in requested or v in requested:
            keep_idx.append(i)
    if not keep_idx:
        raise ValueError(f"No requested members found. requested={members}, available sample={member_vals[:5]}")
    return da.isel(member=keep_idx)


def load_case_o3_for_evolution(case):
    da, date = load_hindcast_o3(case)
    da = da.assign_coords(date=("lead_time", date))
    return da, date


def load_ref_o3_curve(year=REF_YEAR):
    da, date = load_bwcn_ref_o3(year)
    return da, date


def load_b2000_o3_climatology():
    path = B2000_ROOT / "partial_O3" / "B2000WCN_partial_O3_all_ranges.nc"
    if not path.exists():
        print(f"Missing B2000 O3 climatology source: {path}")
        return None
    with xr.open_dataset(path, decode_times=False) as ds:
        da = ds["O3_partial_60_90N_30_70hPa"]
        date = np.asarray(ds["date"].values, dtype=np.int64)
        _, mm, dd = date_parts(date)
        mmdd = np.array([m * 100 + d for m, d in zip(mm, dd)])
        da = da.assign_coords(mmdd=("time", mmdd))
        clim = da.groupby("mmdd").mean("time", skipna=True).load()
    all_mmdd = clim["mmdd"].values
    mask = (all_mmdd >= 101) & (all_mmdd <= 530)
    clim = clim.isel(mmdd=mask).rename({"mmdd": "lead_time"}).assign_coords(lead_time=np.arange(mask.sum()))
    return clim


def u60n10_from_file(path: Path, start_end=((1, 1), (5, 30))) -> Tuple[xr.DataArray, np.ndarray]:
    with xr.open_dataset(path, decode_times=False) as ds:
        date = np.asarray(ds["date"].values, dtype=np.int64)
        mask = date_mask(date, start=start_end[0], end=start_end[1])
        ds = ds.isel(time=mask).sel(lat=60.0, method="nearest")
        date = date[mask]
        p_mid = ds["hyam"] * ds["P0"] + ds["hybm"] * ds["PS"]
        u = interp_profile_logp(ds["U"].transpose("time", "lon", "lev"), p_mid.transpose("time", "lon", "lev"), PLEV_U_PA)
        u_zm = u.mean("lon", skipna=True).load()
    u_zm = u_zm.rename({"time": "lead_time"}).assign_coords(lead_time=np.arange(len(date)), date=("lead_time", date))
    return u_zm, date


def build_u60n10_case_cache(case, overwrite=False):
    out = CACHE_DIR / f"U60N10_{case}_members.nc"
    if out.exists() and not overwrite:
        da = xr.open_dataarray(out).load()
        date = np.asarray(da["date"].values, dtype=np.int64)
        return da, date
    files = sorted((HINDCAST_ROOT / case / "U").glob("*.U.nc"))
    if not files:
        raise FileNotFoundError(f"No U files for {case}")
    das, mids = [], []
    date0 = None
    for f in files:
        mid = member_short_id(f.name)
        print("U60N10", case, mid)
        da, date = u60n10_from_file(f)
        das.append(da)
        mids.append(mid)
        date0 = date
    out_da = xr.concat(das, dim=pd.Index(mids, name="member"))
    out_da.name = "U60N10"
    out_da.to_netcdf(out)
    print(f"Saved cache: {out}")
    return out_da, date0


def build_u60n10_ref_cache(year=REF_YEAR, overwrite=False):
    out = CACHE_DIR / f"U60N10_BWCN_{year:04d}.nc"
    if out.exists() and not overwrite:
        da = xr.open_dataarray(out).load()
        return da, np.asarray(da["date"].values, dtype=np.int64)
    f = BWCN_ROOT / "U" / f"BWCN.cam.h3.{year:04d}.U.nc"
    da, date = u60n10_from_file(f)
    da.name = "U60N10"
    da.to_netcdf(out)
    return da, date


def plot_evolution(ref_data, ref_date, clim_data, experiments, ylabel, ylim, title, name, xlim=(0, 150)):
    fig, ax = plt.subplots(figsize=(12, 7), constrained_layout=True)
    xs = np.arange(len(ref_date))
    ax.plot(xs, ref_data.values, color="black", lw=3.5, label="Reference", zorder=10)
    if clim_data is not None:
        ax.plot(np.arange(clim_data.sizes[clim_data.dims[0]]), clim_data.values, color="black", ls=":", lw=3.2, label="Climatology", zorder=9)
    for exp in experiments:
        data, offset, color, label = exp["data"], exp["offset"], exp["color"], exp["label"]
        show_members = exp.get("show_members", True)
        show_spread = exp.get("show_spread", True)
        line_style = exp.get("ls", "-")
        if data is None:
            continue
        total = data.sizes["lead_time"]
        x = np.arange(offset, offset + total)
        keep = (x >= xlim[0]) & (x <= xlim[1])
        x = x[keep]
        sub = data.isel(lead_time=keep)
        if "member" in sub.dims:
            ens = sub.transpose("member", "lead_time")
            mean = ens.mean("member", skipna=True)
            std = ens.std("member", skipna=True)
            if show_members:
                for i in range(ens.sizes["member"]):
                    ax.plot(x, ens.isel(member=i).values, color=color, alpha=0.14, lw=1.0)
            if show_spread:
                ax.fill_between(x, (mean - std).values, (mean + std).values, color=color, alpha=0.20)
            ax.plot(x, mean.values, color=color, lw=3.2, ls=line_style, label=label)
        else:
            ax.plot(x, sub.values, color=color, lw=3.2, ls=line_style, label=label)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_ylabel(ylabel, fontsize=13)
    ax.set_title(title, fontsize=15)
    ax.set_xticks([0, 31, 59, 90, 120, 151])
    ax.set_xticklabels(["Jan", "Feb", "Mar", "Apr", "May", "Jun"])
    ax.legend(fontsize=10)
    savefig(fig, name)
    plt.show()


def load_diag_for_case(diag, case):
    if diag == "O3":
        return load_case_o3_for_evolution(case)[0]
    if diag == "U60N10":
        da, _ = build_u60n10_case_cache(case)
        return da
    raise ValueError(diag)


def plot_jan_initial_best_worst(diag, data, ref, ref_date, clim, ylabel, ylim):
    best_da = subset_members_da(data, best5_members)
    worst_da = subset_members_da(data, worst5_members)
    experiments = [
        {"data": data, "label": "H-JanInit all 30 mean", "offset": 0, "color": "darkorange", "show_members": True, "show_spread": True},
        {"data": best_da, "label": "Best 5 mean", "offset": 0, "color": "dodgerblue", "show_members": False, "show_spread": False},
        {"data": worst_da, "label": "Worst 5 mean", "offset": 0, "color": "crimson", "show_members": False, "show_spread": False},
    ]
    plot_evolution(
        ref,
        ref_date,
        clim,
        experiments,
        ylabel,
        ylim,
        title=f"{diag}: JAN INITIAL ONLY all/best5/worst5 by O3 RMSE",
        name=f"{diag}_{CASE}_JAN_INITIAL_ONLY_all_best5_worst5_mean",
    )


evolution_specs = [
    {
        "name": "INT3D_vs_CLIM3D_0008_FebInit",
        "experiments": [
            ("0008-02", "H-INT-3D", 31, "forestgreen"),
            ("0008-02_NOCOUPL", "H-CLIM-3D", 31, "royalblue"),
        ],
    },
    {
        "name": "JAN_INITIAL_ONLY_0008",
        "experiments": [
            ("0008-01", "H-JanInit", 0, "darkorange"),
        ],
    },
]

for diag in ["O3", "U60N10"]:
    if diag == "O3":
        ref, ref_date = load_ref_o3_curve(REF_YEAR)
        clim = load_b2000_o3_climatology()
        ylabel, ylim = "Partial O3 column, 30-70 hPa (DU)", (60, 150)
    else:
        ref, ref_date = build_u60n10_ref_cache(REF_YEAR)
        old_clim_path = Path("/home/weiji/restart_exam/code/20260415egu/plots/hindcast/O3/cache/U60N10/U60N10_B2000WCN_climatology.nc")
        clim = xr.open_dataarray(old_clim_path).load() if old_clim_path.exists() else None
        ylabel, ylim = "Zonal-mean U at 60N, 10 hPa (m s-1)", (-50, 90)
    for spec in evolution_specs:
        experiments = []
        jan_initial_data = None
        for case, label, offset, color in spec["experiments"]:
            try:
                data = load_diag_for_case(diag, case)
            except FileNotFoundError as exc:
                print(f"Skip {diag} {case}: {exc}")
                data = None
            if case == CASE:
                jan_initial_data = data
            experiments.append({"data": data, "label": label, "offset": offset, "color": color})
        plot_evolution(
            ref, ref_date, clim, experiments, ylabel, ylim,
            title=f"{diag}: {spec['name']}",
            name=f"{diag}_{spec['name']}",
        )
        if spec["name"] == "JAN_INITIAL_ONLY_0008" and jan_initial_data is not None:
            plot_jan_initial_best_worst(diag, jan_initial_data, ref, ref_date, clim, ylabel, ylim)


## Follow-up Diagnostics: Combined Long Waves And Source Tests

These blocks extend the first pass after the initial run:

- Add `wave1 + wave2` to the main EPFlux/RMSE wave-component figure.
- Keep all RMSE analyses on the full 30-member ensemble.
- Put JAN INITIAL ONLY all-30, best-5, and worst-5 mean evolution curves in the evolution block.
- Split Z300 source tests into fixed-object figures: ACC, stationary-wave closeness, stationary-wave projection, and their RMSE links.
- Add Z300 pointwise correlation maps and zonal wavenumber-amplitude tests as source-hunting diagnostics.


## 图：Z300 monthly pointwise correlation maps

**做什么**：分别为 Jan、Feb、Mar、Apr、May 画五张 Z300 pointwise correlation map。每张图包含六个子图：同月 EP100 all waves、wave1、wave2、wave1+wave2、wave rest，以及整季 O3 RMSE。

**怎么做**：每个格点上，用 30 个成员的 Z300 monthly anomaly（先减 ensemble mean）与一个成员标量做相关。EP100 指标是同月 `mean(-ep2)`，即 100 hPa、40-80N 平均的垂直 EP-flux 分量，不是 divergence。RMSE 仍是 `60-90N, 30-70 hPa` O3 的 Jan1-May30 RMSE。底图使用与 `WindEpfluxEddyheatflux_Structure.ipynb` 一致的 `NorthPolarStereo`，填色为相关系数，黑色等值线为 B2000WCN001002 同月 300 hPa climatological stationary waves（Z300 气候态去 zonal mean）。

**科学问题**：哪些对流层 300 hPa 区域的成员间高度异常，与同月进入平流层的 wave1/wave2/wave-rest EPFlux 或最终 O3 RMSE 同变？这些局地相关是否贴近 climatological stationary-wave ridge/trough？

**预期**：如果对流层 stationary-wave geometry 是源头，相关高值应在气候态 stationary-wave 大振幅区附近成片出现，并且 W1+W2 图应比单独 wave1/wave2 更接近 all waves。

**运行后解读**：待图生成后填写。


In [ ]:
# -----------------------------
# Source-hunting idea 1: monthly Z300 pointwise correlation maps across members
# Colors are local member-to-member correlations. Black contours are the B2000WCN001002
# climatological stationary waves at 300 hPa for the same month/window.
# -----------------------------

def pointwise_member_corr(field_member_lat_lon: xr.DataArray, metric_by_member: pd.Series) -> xr.DataArray:
    da = field_member_lat_lon.copy()
    member_short = [member_short_id(v) for v in da["member"].values]
    da = da.assign_coords(member_short=("member", member_short)).swap_dims({"member": "member_short"})
    common = [m for m in da["member_short"].values if m in metric_by_member.index]
    da = da.sel(member_short=common)
    x = np.asarray([metric_by_member.loc[m] for m in common], dtype=float)
    x = xr.DataArray(x, dims="member_short", coords={"member_short": common})
    x_anom = x - x.mean("member_short")
    y_anom = da - da.mean("member_short", skipna=True)
    cov = (x_anom * y_anom).mean("member_short", skipna=True)
    corr = cov / (x_anom.std("member_short") * y_anom.std("member_short", skipna=True))
    corr.name = "member_correlation"
    return corr


try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    from cartopy.util import add_cyclic_point

    HAS_CARTOPY = True
except ImportError:
    HAS_CARTOPY = False
    ccrs = None
    cfeature = None
    add_cyclic_point = None


def map_lon_lat_values(da: xr.DataArray):
    """Return 2-D lat-lon data in a Cartopy-safe -180..180 longitude convention."""
    lon = np.asarray(da["lon"].values, dtype=float)
    lat = np.asarray(da["lat"].values, dtype=float)
    values = np.asarray(da.transpose("lat", "lon").values, dtype=float)
    if lat[0] > lat[-1]:
        lat = lat[::-1]
        values = values[::-1, :]

    lon_wrapped = ((lon + 180.0) % 360.0) - 180.0
    order = np.argsort(lon_wrapped)
    lon_sorted = lon_wrapped[order]
    values_sorted = values[:, order]

    if add_cyclic_point is None:
        return lon_sorted, lat, values_sorted
    try:
        values_cyc, lon_cyc = add_cyclic_point(values_sorted, coord=lon_sorted)
        return lon_cyc, lat, values_cyc
    except ValueError:
        return lon_sorted, lat, values_sorted


def add_polar_map_features(ax, data_crs):
    try:
        ax.add_feature(cfeature.COASTLINE.with_scale("110m"), linewidth=0.55, edgecolor="0.25")
        ax.add_feature(cfeature.BORDERS.with_scale("110m"), linewidth=0.28, edgecolor="0.35", alpha=0.75)
        ax.gridlines(crs=data_crs, linewidth=0.32, color="0.45", alpha=0.45, linestyle="--", draw_labels=False)
    except Exception as exc:
        ax.text(
            0.02,
            0.02,
            f"map features unavailable: {type(exc).__name__}",
            transform=ax.transAxes,
            fontsize=7,
            ha="left",
            va="bottom",
            color="0.35",
        )


def plot_corr_map_with_stationary_contours(
    corr_da: xr.DataArray,
    stationary_da: xr.DataArray,
    ax,
    title,
    vlim=0.8,
    wave_levels=np.arange(-240, 241, 40),
):
    lon, lat, corr_values = map_lon_lat_values(corr_da)
    _, _, wave_values = map_lon_lat_values(stationary_da)
    levels = np.linspace(-vlim, vlim, 17)
    if HAS_CARTOPY:
        data_crs = ccrs.PlateCarree()
        cf = ax.contourf(lon, lat, corr_values, levels=levels, cmap="RdBu_r", extend="both", transform=data_crs)
        ax.contour(lon, lat, wave_values, levels=wave_levels, colors="k", linewidths=0.62, alpha=0.84, transform=data_crs)
        ax.set_extent([-180.0, 180.0, LAT_Z300[0], LAT_Z300[1]], crs=data_crs)
        add_polar_map_features(ax, data_crs)
    else:
        cf = ax.contourf(lon, lat, corr_values, levels=levels, cmap="RdBu_r", extend="both")
        ax.contour(lon, lat, wave_values, levels=wave_levels, colors="k", linewidths=0.62, alpha=0.84)
        ax.set_xlim(-180, 180)
        ax.set_ylim(*LAT_Z300)
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")
    ax.set_title(title, fontsize=8)
    return cf


def monthly_pointwise_metrics(month, start_end):
    ep_month = ep_wide_for_window(start_end)
    metrics = ep_month.merge(rmse_ep_all[["member", "RMSE_DU"]], on="member", how="left")
    metrics["EP100_wave1_plus_wave2"] = metrics["EP100_wave1"] + metrics["EP100_wave2"]
    z300_month = build_z300_hindcast_cache(CASE, start_end=start_end, label=month)
    z300_anom = z300_month - z300_month.mean("member", skipna=True)
    stationary = build_z300_b2000_stationary_target(month, start_end)
    corr_maps = {
        col: pointwise_member_corr(z300_anom, metrics.set_index("member")[col])
        for col in ["EP100_all_waves", "EP100_wave1", "EP100_wave2", "EP100_wave1_plus_wave2", "EP100_wave_rest", "RMSE_DU"]
    }
    ds_maps = xr.Dataset({col: da for col, da in corr_maps.items()})
    out = CACHE_DIR / f"Z300_pointwise_corr_maps_{CASE}_{window_token(month)}.nc"
    ds_maps.to_netcdf(out)
    print("Saved:", out)
    return corr_maps, stationary


def plot_monthly_pointwise_corr_maps(month, start_end):
    corr_maps, stationary = monthly_pointwise_metrics(month, start_end)
    metric_cols = ["EP100_all_waves", "EP100_wave1", "EP100_wave2", "EP100_wave1_plus_wave2", "EP100_wave_rest", "RMSE_DU"]
    titles = ["All waves", "Wave 1", "Wave 2", "Wave 1 + Wave 2", "Wave rest", "O3 RMSE"]
    subplot_kw = {"projection": ccrs.NorthPolarStereo()} if HAS_CARTOPY else {}
    fig, axes = plt.subplots(2, 3, figsize=(11.8, 8.4), constrained_layout=True, subplot_kw=subplot_kw)
    cf = None
    for ax, col, title in zip(axes.ravel(), metric_cols, titles):
        cf = plot_corr_map_with_stationary_contours(corr_maps[col], stationary, ax, title)
    cbar = fig.colorbar(cf, ax=axes.ravel().tolist(), shrink=0.76, pad=0.02)
    cbar.set_label("Correlation r", fontsize=8)
    cbar.ax.tick_params(labelsize=8)
    fig.suptitle(
        f"{CASE} {month}: Z300 pointwise r; contours = B2000 stationary waves",
        fontsize=10,
    )
    savefig(fig, f"{CASE}_Z300_pointwise_corr_maps_{window_token(month)}_EPFlux_RMSE")
    plt.show()


for month in MONTH_ORDER:
    plot_monthly_pointwise_corr_maps(month, MONTH_WINDOWS[month])


## 图：Z300 zonal wavenumber amplitude

**做什么**：把 Jan20-Feb10 的 Z300 stationary anomaly 按经向 Fourier 分解，计算 wave1、wave2、wave3-6 振幅，并与 EPFlux/O3 RMSE 比较。

**怎么做**：先对每个成员的 300 hPa 高度场去掉 zonal mean，再沿经度取 Fourier wave-k 振幅，最后在 20-90N 做 cos-lat 加权平均。`synoptic_3_6` 是 wave3-6 振幅平方和开根号。

**科学问题**：这是一个“对流层波源/波型强度”诊断：成员间对流层 300 hPa 的长波振幅，是否已经预示了后续 100 hPa EPFlux 的 wave1/wave2 差异？它回答的是源头波型是否存在，而不是波活动是否真的向上传播。

**和直接比较 300 hPa EPFlux Fz vs 100 hPa EPFlux Fz 的区别**：300-to-100 hPa EPFlux 对比更直接检验垂直传播/耦合链条；Z300 wave amplitude 则更上游，检验对流层 stationary-wave 几何结构和振幅本身。前者更接近“波活动是否传上去”，后者更接近“对流层有没有提供 wave1/wave2 源”。两者应该并行使用：若 Z300 wave2 amp 与 100 hPa wave2 EPFlux 相关，而 300 hPa EPFlux 与 100 hPa EPFlux 也相关，证据链会更完整。

**论文依据**：行星波从对流层进入冬季平流层的基本传播理论可参考 Charney and Drazin (1961), DOI `10.1029/JZ066i001p00083`；stationary planetary wave 垂直传播与极涡扰动可参考 Matsuno (1970), DOI `10.1175/1520-0469(1970)027<0871:VPOSPW>2.0.CO;2`；stationary wave activity flux 诊断可参考 Plumb (1985), DOI `10.1175/1520-0469(1985)042<0217:OTTDPO>2.0.CO;2`；上行 wave activity flux 作为极端平流层事件前兆可参考 Polvani and Waugh (2004), DOI `10.1175/1520-0442(2004)017<3548:UWAFAA>2.0.CO;2`。

**预期**：已有初步结果显示 Z300 wave2 amplitude 与 EP100 wave2 相关最强；这支持 wave2 相关源头假设，但还不直接证明因果。

**运行后解读**：待图生成后填写。


In [ ]:
# -----------------------------
# Source-hunting idea 2: Z300 zonal-wavenumber amplitudes
# Wave-k amplitude is computed from the C-window 300 hPa height field after removing the zonal mean,
# using longitude Fourier coefficients at each latitude and cos-lat averaging over 20-90N.
# Synoptic 3-6 amplitude is sqrt(wave3^2 + wave4^2 + wave5^2 + wave6^2).
# EP100 is the same scalar -ep2 metric: 100 hPa, 40-80N, Jan20-Feb10; it is not divergence.
# -----------------------------

def z300_wave_amplitude(da: xr.DataArray, k: int, lat_range=LAT_Z300) -> float:
    sub = select_latband(da, lat_range)
    arr = sub.values
    lon = np.deg2rad(sub["lon"].values)
    cos_term = np.cos(k * lon)
    sin_term = np.sin(k * lon)
    a = np.nanmean(arr * cos_term[None, :], axis=1) * 2.0
    b = np.nanmean(arr * sin_term[None, :], axis=1) * 2.0
    amp_lat = np.sqrt(a * a + b * b)
    w = np.cos(np.deg2rad(sub["lat"].values)).clip(0, 1)
    return float(np.nansum(amp_lat * w) / np.nansum(w))

z_rows = []
for mid in z300_members["member"].values:
    z = z300_members.sel(member=mid)
    z_anom_lon = z - z.mean("lon", skipna=True)
    row = {"member": member_short_id(mid)}
    for k in [1, 2, 3, 4, 5, 6]:
        row[f"Z300_wave{k}_amp"] = z300_wave_amplitude(z_anom_lon, k)
    row["Z300_synoptic_3_6_amp"] = np.sqrt(sum(row[f"Z300_wave{k}_amp"] ** 2 for k in [3, 4, 5, 6]))
    z_rows.append(row)
z_wave_df = pd.DataFrame(z_rows)
z_wave_join = rmse_ep_all.merge(z_wave_df, on="member", how="left")
z_wave_join["EP100_wave1_plus_wave2"] = z_wave_join["EP100_wave1"] + z_wave_join["EP100_wave2"]
z_wave_join.to_csv(TABLE_DIR / f"{CASE}_Z300_wave_amplitude_metrics.csv", index=False)

amp_corr_rows = []
for amp_col in ["Z300_wave1_amp", "Z300_wave2_amp", "Z300_synoptic_3_6_amp"]:
    for target_col in ["EP100_all_waves", "EP100_wave1", "EP100_wave2", "EP100_wave1_plus_wave2", "EP100_wave_rest", "RMSE_DU"]:
        r, p = finite_corr(z_wave_join[amp_col], z_wave_join[target_col])
        amp_corr_rows.append({"z300_metric": amp_col, "target": target_col, "R": r, "P": p})
amp_corr = pd.DataFrame(amp_corr_rows)
amp_corr.to_csv(TABLE_DIR / f"{CASE}_Z300_wave_amplitude_correlations.csv", index=False)
print(amp_corr.sort_values("P").head(12))

fig, axes = plt.subplots(2, 3, figsize=(16, 9), constrained_layout=True)
plot_pairs = [
    ("Z300_wave1_amp", "EP100_wave1"),
    ("Z300_wave2_amp", "EP100_wave2"),
    ("Z300_wave1_amp", "EP100_wave1_plus_wave2"),
    ("Z300_synoptic_3_6_amp", "EP100_wave_rest"),
    ("Z300_wave1_amp", "RMSE_DU"),
    ("Z300_synoptic_3_6_amp", "RMSE_DU"),
]
for ax, (xcol, ycol) in zip(axes.ravel(), plot_pairs):
    scatter_fit(
        ax,
        z_wave_join,
        xcol,
        ycol,
        f"{xcol} vs {ycol}",
        xlabel=xcol,
        ylabel=ycol,
        color="tab:brown",
        annotate_members=False,
    )
fig.suptitle(
    "Z300 zonal-wavenumber amplitudes: Fourier amplitude of C-window 300 hPa height stationary anomalies",
    fontsize=11,
)
savefig(fig, f"{CASE}_Z300_wave_amplitudes_vs_EPFlux_RMSE")
plt.show()

## Notes For Interpretation

- O3 RMSE uses the cleaned `partial_O3` product: polar-cap `60-90N`, `30-70 hPa`, compared against BWCN year 0008.
- Every `EP100_*` scalar is `mean(-ep2)` from `EPflux_daily_ubar`: vertical EP-flux component at `100 hPa`, cos-lat averaged over `40-80N`, then averaged over the stated window. It is not `div1`, `div2`, or EP-flux divergence. The sign is flipped so positive follows the old upward-Fz convention.
- All RMSE/EPFlux scatter tests use all 30 members.
- The asynchronous AO test uses B2000WCN-projected AO averaged over a candidate window and compares it with EP100 wave activity in an equal-length window shifted later by a positive delay; the fixed diagnostic is AO Jan10-Feb08 vs EPFlux Jan20-Feb18, both 30 days.
- AO and NAM are both based on zonal-mean WACCM logic; AO is the 1000 hPa zonal-mean Code-B modified projection.
- Z300 ACC compares each member to BWCN0008 for the same window/month. Stationary-wave closeness and stationary-wave projection use the B2000WCN001002 all-year climatological stationary-wave target for the same window/month, computed from raw Z3 and cached when needed.
- The pointwise Z300 maps correlate local member-to-member Z300 anomalies with scalar EPFlux/RMSE metrics; they do not plot longitude-resolved EPFlux.
- Z300 wave-k amplitude is the longitude Fourier amplitude of the `Jan20-Feb10` 300 hPa stationary anomaly, cos-lat averaged over `20-90N`; `synoptic_3_6` combines waves 3-6 in quadrature.
